# Replicating direct weight writing with small neural networks

A neural network makes decisions through calculations whose numerical coefficients are called weights. Usually a training program chooses those numbers by repeatedly adjusting mistakes on examples. Another possibility is to ask a language model to choose the actual coefficients directly. A useful outcome would be a complete set of numbers that makes the network perform the requested task on inputs it has not been shown.

This notebook is part of our attempt to reproduce that idea on much smaller models, motivated by [Sparks of Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/) (Takehi et al., 2026). The distinction is between writing the numbers and merely writing code that trains a network. Throughout these experiments, the language model authors the candidate parameters and an independent harness saves and tests them unchanged.

Start with an explicitly illustrative task: two numbers belong to class one when their signs differ, and class zero when their signs match. A human can describe that rule in one sentence. A language model that writes weights must go further: its numbers have to make the network's arithmetic distinguish the two cases. An answer that explains the rule well but supplies unusable coefficients has not completed the task.

One complication is a curved decision boundary. Points can be assigned to rings according to distance from the origin, but a network built from straight-line calculations has to approximate the curves. Another complication is learning the rule from examples rather than receiving it explicitly. These changes help distinguish a compact numerical implementation of a known rule from choosing a useful classifier from a labelled sample.

Our experiments therefore move from seventeen-value rules to a 388-value ring classifier, then to a 6,570-value handwritten-digit classifier. The digit input is an eight-by-eight grid of ink counts, and the answer is one of ten labels. For an illustrative zero, a loop of dark blocks surrounds a blank centre. A six may also contain a loop, so one pixel detector will not settle every case. The network needs combinations of evidence that remain useful when handwriting changes.

The architecture specifies which calculations happen. Each layer multiplies inputs by a weight matrix, adds biases and passes its results onward. The final layer produces scores, and the largest score identifies the predicted class. The teacher receives this exact arithmetic and must supply every coefficient. It cannot execute code, inspect files, run an optimizer or query the evaluator while writing its answer.

The linked [SQL experiment](../42-tiny-sql-direct-weights/) tests another branch of the same replication attempt: directly writing a tiny sequence generator.

The central question is how far the direct-writing result extends when the network is smaller and the task changes. A long numerical response can also fail before its function can be tested, and the public digit dataset may be familiar from pretraining, so a score shows what the submitted network does on these images, not how much the teacher learned from the sample.

## Study summary

This component of the replication tests directly written MLPs at three sizes: seventeen parameters for explicit mathematical rules, 388 for four rings, and **6,570 for digits**. Astra succeeds on the small rule tests; the larger comparison gives twelve models the same **2,048 labelled images** and measures performance on **1,797 official test images**. Another 1,775 training images are held back as a secondary check. An MLP, or multilayer perceptron, is a network of successive fully connected layers; each neuron in a layer receives every output of the previous layer.

Accuracy counts the images whose predicted digit equals the supplied label: 90 correct among 100 images is 90%, and random guessing gives about 10%. A one-nearest-neighbour reference, which copies the label of the most similar visible image, scores 1,750/1,797 (97.38%); it is a separate method, not a source of network weights.

Twelve models received thirteen first requests (Opus 5.5 twice, at two settings). Of the ten asked for a flat list of numbers, **four returned a network that could be scored**; all three open-weight models, asked for numbered entries from the start, did. Counting numbered-entry retries, ten networks could be scored in all. Best first, they classified this many of the 1,797 test images correctly: Astra 1,645 (91.54%); Fable 5.1 1,628 (90.60%); Opus 5.5 high (numbered entries) 1,580 (87.92%); Sol 1,422 (79.13%); Sonnet 5.5 (numbered entries) 1,214 (67.56%); Gemini batch (numbered entries) 1,091 (60.71%); Muse Wonder 606 (33.72%); DeepSeek V4.1 Flash 379 (21.09%); MiMo-V2.6-Pro 335 (18.64%); GLM-5.3 194 (10.80%). The other 11 submissions produced no network to score, and they are counted apart from wrong digits.

Fable 5.1 is the one Claude model whose first flat reply could be scored: exactly 6,570 values, 1,628 correct (90.60%), after 166,646 thinking tokens and US$19.79. Most other failures were failures to write the numbers out, not bad designs. Opus 5.5 and Sonnet 5.5 each wrote a template-matching classifier in their first flat reply and then miscounted rows: Opus left out three rows of zeros (6,378 values) and Sonnet added a line of 56 zeros and dropped two biases (6,624). Asked again for numbered entries per tensor, each returned the same values with correct counts in one reply. Gemini needed three unusable requests and two numbered replies. Haiku 5.5 ran into long runs of zeros twice, then declined to write weights it could not test. The first Opus request, at the highest effort setting, spent its whole 64,000-token allowance thinking and wrote nothing.

Asking for numbered entries from the start removed the format problem but not the design problem. The three highest-ranked open-weight models on OpenRouter each delivered all 6,570 entries; GLM-5.3 and MiMo-V2.6-Pro needed a second call because their first ran out of output, or hit a provider error, while still reasoning. Their networks classify 21.1% (DeepSeek V4.1 Flash), 10.8% (GLM-5.3) and 18.6% (MiMo-V2.6-Pro) of the test images correctly, because none of them puts its ten digit scores on a common scale, so one or two digits win almost every image.

This is one submission per model and setting on an old public benchmark. Settings are not matched: the Claude requests used high effort and a 128,000-token allowance, the open-weight models medium reasoning with numbered entries asked for from the start, the other models their providers' settings, and the numbered retries saw their own earlier reply. Possible pretraining exposure makes every score an upper bound on learning from unseen data, and 1,797 test images do not turn ten single networks into a reliable model ranking.

## Contents

- [1. Replication scope and data](#1-replication-scope-and-data)
- [2. The fixed network and numerical contract](#2-the-fixed-network-and-numerical-contract)
- [3. Model access and isolation](#3-model-access-and-isolation)
- [4. How the study got here](#4-how-the-study-got-here)
- [5. Results and comparisons](#5-results-and-comparisons)
- [6. Worked test cases and error analysis](#6-worked-test-cases-and-error-analysis)
- [7. Reasoning logs and parameter patterns](#7-reasoning-logs-and-parameter-patterns)
- [8. Limits and practical meaning](#8-limits-and-practical-meaning)
- [9. Reproduction, caches and spend](#9-reproduction-caches-and-spend)
- [10. Future work and further reading](#10-future-work-and-further-reading)

```mermaid
flowchart LR
    U["UCI optical digits<br/>5,620 labelled 8×8 images"] --> S["Freeze official split and sample<br/>2,048 visible; 3,572 hidden"]
    S --> P["Identical text packet<br/>visible CSV + exact inference code"]
    P --> M["13 first requests to 12 models + format retries<br/>each asks for 6,570 numbers"]
    M --> V["Strict parsing and unchanged loading<br/>readable network or separate failure"]
    V --> E["Fixed float64 forward pass<br/>1,797 test + 1,775 unused training cases"]
    E --> R["Accuracy, confusion, examples<br/>time, cost and emitted reasoning"]
```

One visible image travels as sixty-four ink counts and a digit label in the packet that goes to every teacher. A returned parameter vector is reshaped into six tensors and applied to hidden images, whose labels enter only when predictions are scored. No teacher sees another's output or any score.

### Replay setup

The next cell defines this notebook's helpers in place. It does not import experiment code from another folder and does not invoke a teacher. Ordinary execution uses only committed data.

In [1]:
"""Direct numeric weight submissions only. No training or tester-authored candidate weights."""
from __future__ import annotations
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime as dt
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time
import zipfile
import numpy as np
import requests

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA = ROOT / 'data'
MODELS = {
    'astra': {'backend':'codex', 'model':'gpt-6-astra', 'effort':'xhigh'},
    'sol': {'backend':'codex', 'model':'gpt-6.1-sol', 'effort':'xhigh'},
    'opus': {'backend':'claude', 'model':'claude-opus-5-5', 'effort':'xhigh'},
    'muse': {'backend':'muse', 'model':'muse-wonder-full-1.0', 'effort':'xhigh'},
    'nemotron': {'backend':'openrouter', 'model':'nvidia/nemotron-3.5-super-vl:free', 'canonical':'nvidia/nemotron-3.5-super-vl-20261015'},
    'gemini': {'backend':'batch', 'model':'google/gemini-4-argon:batch', 'canonical':'google/gemini-4-argon-20261013', 'effort':'high'},
    # Claude family requests (data/claude_family/), collected by claude_family.py at high effort.
    'sonnet': {'backend':'claude', 'model':'claude-sonnet-5-5', 'effort':'high'},
    'haiku': {'backend':'claude', 'model':'claude-haiku-5-5', 'effort':'high'},
    'fable': {'backend':'claude', 'model':'claude-fable-5-1', 'effort':'high'},
    # Open-weight models asked for numbered entries from the start (data/open_models/), collected by open_models.py.
    'deepseek': {'backend':'openrouter', 'model':'deepseek/deepseek-v4.1-flash-1017', 'canonical':'deepseek/deepseek-v4.1-flash-20261017', 'provider':'deepseek', 'effort':'medium'},
    'glm': {'backend':'openrouter', 'model':'z-ai/glm-5.3', 'canonical':'z-ai/glm-5.3-20260816', 'provider':'z-ai', 'effort':'medium'},
    'mimo': {'backend':'openrouter', 'model':'xiaomi/mimo-v2.6-pro', 'canonical':'xiaomi/mimo-v2.6-pro-20260921', 'provider':'xiaomi', 'effort':None},
}
FIRST_ROUND = ['astra', 'sol', 'opus', 'muse', 'nemotron', 'gemini']
CLAUDE_FAMILY = ['opus', 'sonnet', 'haiku']
N_VISIBLE = 2048
MAX_NEW_CALLS = 1
MAX_OUTPUT_TOKENS = 131072
CLAUDE_MAX_BUDGET_USD = 5.0
SEED = 20261103
SHAPES = {'W1':[64,64], 'b1':[64], 'W2':[32,64], 'b2':[32], 'W3':[10,32], 'b3':[10]}
N_PARAMS = sum(int(np.prod(s)) for s in SHAPES.values())
assert N_PARAMS == 6570
SCHEMA = {'type':'object', 'additionalProperties':False, 'required':['weights'], 'properties':{
    'weights':{'type':'array','minItems':N_PARAMS,'maxItems':N_PARAMS,'items':{'type':'number'}}}}
PREPOST = '''import numpy as np
def preprocess(raw_pixels):
    # Exactly 64 integers, row-major flattening of the 8 by 8 image.
    # 0 is blank, 16 is the largest ink count. No fitted transform.
    return np.asarray(raw_pixels, dtype=np.float64) / 16.0
def network(raw_pixels, W1, b1, W2, b2, W3, b3):
    x = preprocess(raw_pixels)
    h1 = np.maximum(x @ W1.T + b1, 0.0)
    h2 = np.maximum(h1 @ W2.T + b2, 0.0)
    return h2 @ W3.T + b3
def postprocess(scores):
    # Digit labels are 0 through 9. Ties choose the lowest index.
    return np.argmax(scores, axis=-1)
'''

def dump(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

def sha(value):
    return hashlib.sha256(value if isinstance(value,bytes) else value.encode()).hexdigest()

def utc():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def trace_call(key, config, request, record):
    """One trace per fresh call, silently optional, never called for cache replay."""
    host = os.environ.get('LANGFUSE_HOST','http://localhost:3300').rstrip('/')
    auth = (os.environ.get('LANGFUSE_PUBLIC_KEY',''),os.environ.get('LANGFUSE_SECRET_KEY',''))
    try:
        healthy = requests.get(host+'/api/public/health',timeout=2).ok
    except requests.RequestException:
        healthy = False
    if not healthy or not all(auth): return
    def attr(k,v): return {'key':k,'value':{'stringValue':v if isinstance(v,str) else json.dumps(v,default=str)}}
    notebook = '41-direct-weight-mlps'
    attrs = [attr('langfuse.observation.type','generation'),attr('langfuse.observation.model.name',config['model']),
             attr('langfuse.observation.input',request),attr('langfuse.observation.output',record.get('answer','')),
             attr('langfuse.trace.name','41 direct MLP weights'),attr('langfuse.environment','notebook'),
             attr('langfuse.trace.metadata.notebook',notebook),attr('langfuse.trace.metadata.cache_key',key),
             attr('langfuse.observation.metadata.native_usage',record.get('usage'))]
    if record.get('cost_usd') is not None: attrs.append(attr('langfuse.observation.cost_details',{'total':record['cost_usd']}))
    start = dt.datetime.fromisoformat(record['started_utc']); end = dt.datetime.fromisoformat(record['finished_utc'])
    span = {'traceId':sha(notebook+'|'+key)[:32], 'spanId':sha(notebook+'|'+key+'|gen|0')[:16],
            'name':'Write 6570 MLP parameters','kind':1,'attributes':attrs,
            'startTimeUnixNano':str(int(start.timestamp()*1e9)),'endTimeUnixNano':str(int(end.timestamp()*1e9))}
    try:
        requests.post(host+'/api/public/otel/v1/traces',json={'resourceSpans':[{'resource':{'attributes':[
            attr('service.name','experiment-notebooks')]},'scopeSpans':[{'scope':{'name':notebook},'spans':[span]}]}]},auth=auth,timeout=10)
    except requests.RequestException: pass

def prepare():
    DATA.mkdir(parents=True,exist_ok=True)
    source = DATA/'source'; source.mkdir(exist_ok=True)
    url = 'https://archive.ics.uci.edu/static/public/80/optical+recognition+of+handwritten+digits.zip'
    archive = source/'optdigits.zip'
    if not archive.exists():
        response = requests.get(url,timeout=45); response.raise_for_status(); archive.write_bytes(response.content)
    with zipfile.ZipFile(archive) as z:
        for name in ['optdigits.tra','optdigits.tes','optdigits.names']:
            (source/name).write_bytes(z.read(name))
    train = np.loadtxt(source/'optdigits.tra',delimiter=',',dtype=np.int64)
    test = np.loadtxt(source/'optdigits.tes',delimiter=',',dtype=np.int64)
    assert train.shape == (3823,65) and test.shape == (1797,65)
    rng=np.random.default_rng(SEED); selected=[]
    for label in range(10):
        selected.extend(rng.permutation(np.flatnonzero(train[:,-1]==label))[:N_VISIBLE//10+(label<N_VISIBLE%10)])
    selected=np.asarray(selected);selected=selected[rng.permutation(len(selected))]
    remaining=np.setdiff1d(np.arange(len(train)),selected)
    np.savez_compressed(DATA/'dataset.npz',visible=train[selected],visible_indices=selected,
                        hidden_train=train[remaining],hidden_train_indices=remaining,test=test)
    sample='\n'.join(','.join(map(str,row)) for row in train[selected])+'\n'
    (DATA/'visible_sample.csv').write_text(sample,encoding='utf-8')
    (DATA/'pre_post_processing.py.txt').write_text(PREPOST,encoding='utf-8')
    prompt='''You are the sole numerical weight author for a neural-network classifier.
You have NO tools, code execution, calculator, shell, file access, web, skills, plugins,
MCP or other agents. Reason internally. The tester loads your numerical output unchanged.
It will not train, optimize, correct, fill, truncate or choose any candidate parameter.

TASK: Recognize handwritten digits 0 through 9 from 8 by 8 grayscale ink-count images.
Each CSV line below has 64 pixel values in row-major order, then its integer digit label.
Pixels are integers 0..16, where 0 is blank. The 2048 lines are the ONLY labelled sample
you receive. They were selected, approximately balanced by digit, from the UCI Optical
Recognition of Handwritten Digits official training split. You will be evaluated on its
withheld 1797-image official test split and the 1775 training images not shown here.
Do not output predictions, code or a trained-model download. Your complete numbers must
implement a useful classifier in the fixed MLP below, with no external lookup at inference.

ARCHITECTURE: 64 -> 64 ReLU -> 32 ReLU -> 10 linear scores, float64 computation.
There are exactly 6570 independent editable weights and biases. No other inference
operations are permitted besides the exact preprocessing and postprocessing below.
Sparse constructions and zero biases are allowed; there is no density or Gaussianity
requirement. Prefer numerically moderate coefficients, but every finite value is allowed.

EXACT PREPROCESSING, FORWARD COMPUTATION AND POSTPROCESSING (read, do not execute):
'''+PREPOST+'''
RETURN ONLY {"weights":[6570 explicit finite JSON numbers]}.
Flatten tensors in the following order; flatten each tensor in C/row-major order.
W1 [64,64]: offsets 0..4095, then b1 [64]: 4096..4159;
W2 [32,64]: 4160..6207, then b2 [32]: 6208..6239;
W3 [10,32]: 6240..6559, then b3 [10]: 6560..6569.
Rows of a weight matrix are output neurons; columns are input neurons.
Return ALL 6570 numbers, including zeros. No prose, Markdown, code, formulas, ellipses,
repetition notation, omitted entries, extra keys, or special NaN/Infinity values.
There is one submission and no intermediate feedback. Maximize held-out accuracy.

LABELLED SAMPLE CSV (columns p00,p01,...,p63,label):
'''+sample+'\nEND SAMPLE. Return only the complete JSON weights object.\n'
    (DATA/'prompt.txt').write_text(prompt,encoding='utf-8');dump(DATA/'weights_schema.json',SCHEMA)
    dataset_keys={tuple(row[:64]) for row in train[selected]}
    manifest={'protocol':'digits-v1','prepared_utc':utc(),'source_url':url,
        'source_page':'https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits',
        'creators':'E. Alpaydin and C. Kaynak','source_donation_date':'1998-06-30','licence':'CC BY 4.0',
        'freshness_exception':'User explicitly requested public data. Old public benchmark may have been in teacher training; treat scores as upper bounds.',
        'archive_sha256':sha(archive.read_bytes()),'file_sha256':{name:sha((source/name).read_bytes()) for name in ['optdigits.tra','optdigits.tes','optdigits.names']},
        'architecture':[64,64,32,10],'shapes':SHAPES,'parameters':N_PARAMS,'seed':SEED,
        'n_visible':len(selected),'n_hidden_train':len(remaining),'n_official_test':len(test),
        'visible_class_counts':np.bincount(train[selected,-1],minlength=10).tolist(),
        'test_class_counts':np.bincount(test[:,-1],minlength=10).tolist(),
        'test_pixel_rows_duplicated_in_visible':sum(tuple(row[:64]) in dataset_keys for row in test),
        'prompt_sha256':sha(prompt),'prompt_bytes':len(prompt.encode()),'models':MODELS,
        'no_tools':True,'training_steps':0,'tester_authors_weights':False,'one_submission_per_model':True,
        'primary_metric':'official test accuracy; Wilson 95% interval conditional on this single submitted network',
        'secondary':['visible-sample accuracy','unused-training accuracy','per-class accuracy','confusion matrix','weight statistics'],
        'no_content_retries':True,'no_cancellation_on_task_switch':True,
        'reasoning_policy':'Save raw provider-emitted reasoning, deltas and opaque details; missing reasoning is not reconstructed.',
        'user_authorized_closed_openrouter_exception':'google/gemini-4-argon:batch explicitly requested for this experiment only.'}
    path=DATA/'manifest.json'
    if path.exists(): assert json.loads(path.read_text(encoding='utf-8'))['prompt_sha256']==manifest['prompt_sha256']
    else: dump(path,manifest)
    print(json.dumps({k:manifest[k] for k in ['parameters','n_visible','n_hidden_train','n_official_test','prompt_bytes','test_pixel_rows_duplicated_in_visible']}),flush=True)

def private_dir(key):
    path=Path(tempfile.gettempdir())/'direct-weights-41'/key
    (path/'empty').mkdir(parents=True,exist_ok=True)
    (path/'neutral.txt').write_text('You are a numerical weight author. Return only the requested JSON data.\n',encoding='utf-8')
    dump(path/'empty-mcp.json',{'mcpServers':{}})
    return path

def cli_command(key, out):
    cfg=MODELS[key];private=private_dir(key)
    if cfg['backend']=='codex':
        cmd=json.loads((DATA/'transport'/'codex_template.json').read_text(encoding='utf-8'))
        if '--ephemeral' in cmd:cmd.remove('--ephemeral')
        cmd[cmd.index('-m')+1]=cfg['model'];cmd[cmd.index('-C')+1]=str(private/'empty')
        cmd[cmd.index('--output-schema')+1]=str(DATA/'weights_schema.json')
        for i,arg in enumerate(cmd):
            if arg.startswith('model_reasoning_effort='):cmd[i]='model_reasoning_effort='+json.dumps(cfg['effort'])
            if arg.startswith('model_instructions_file='):cmd[i]='model_instructions_file='+json.dumps(str(private/'neutral.txt'))
        return cmd[:-1]+['-c','model_context_window=1050000','-c','model_auto_compact_token_limit=1000000',
                        '--output-last-message',str(out/'cli_last_message.json'),'-']
    if cfg['backend']=='claude':
        return [shutil.which('claude'),'--print','--model',cfg['model'],'--effort',cfg['effort'],'--tools','',
                '--safe-mode','--disable-slash-commands','--strict-mcp-config','--mcp-config',str(private/'empty-mcp.json'),
                '--disallowedTools','mcp__*','--system-prompt',(private/'neutral.txt').read_text(),
                '--output-format','stream-json','--verbose','--include-partial-messages','--max-budget-usd',str(CLAUDE_MAX_BUDGET_USD)]
    exe=os.environ.get('MUSE_EXECUTABLE') or shutil.which('muse.cmd') or shutil.which('muse')
    return [exe,'exec','--preset','empty','--disable-web-tools','--disable-reminders','--disable-request-user-input',
            '--no-foreign-personal-context','--disable-shell','--disable-write','--model',cfg['model'],
            '--reasoning-effort',cfg['effort'],'--max-model-steps','1','--workspace',str(private/'empty'),
            '--output-schema',str(DATA/'weights_schema.json'),'--prompt-file',str(DATA/'prompt.txt'),'--json']

def parse_cli(backend, events):
    answer='';reasoning=[];usage={};completed=False;tools=[];cost=None;resolved=[];catalogues=[]
    if backend=='codex':
        for e in events:
            if e.get('type')=='item.completed':
                item=e.get('item',{});kind=item.get('type')
                if kind=='agent_message':answer=item.get('text','')
                elif kind=='reasoning' and item.get('text'):reasoning.append(item['text'])
                elif kind not in ['reasoning','error']:tools.append(item)
            if e.get('type')=='turn.completed':usage=e.get('usage',{});completed=True
    elif backend=='claude':
        for e in events:
            if e.get('type')=='system' and e.get('subtype')=='init':
                catalogues.append({k:e.get(k) for k in ['tools','mcp_servers','skills','plugins']});resolved.append(e.get('model'))
            if e.get('type')=='assistant':
                for block in e.get('message',{}).get('content',[]):
                    if block.get('type')=='thinking':reasoning.append(block.get('thinking',''))
                    if block.get('type')=='tool_use':tools.append(block)
            if e.get('type')=='result':
                answer=e.get('result','');usage=e.get('usage',{});cost=e.get('total_cost_usd')
                completed=e.get('subtype')=='success' and not e.get('is_error',False)
    else:
        for e in events:
            kind=e.get('payload_type','');p=e.get('payload',{})
            if kind=='run.terminal.completed':answer=p.get('text','');completed=True
            if kind=='run.usage':usage=p
            if 'reasoning' in kind or 'thinking' in kind:
                reasoning.append(p.get('text') or p.get('delta') or json.dumps(p))
            if kind.startswith('tool.') or 'tool_call' in kind:tools.append(e)
    return dict(answer=answer,reasoning='\n'.join(reasoning),usage=usage,completed=completed,
                tool_events=tools,cost_usd=cost,reported_models=resolved,catalogues=catalogues)

def run_cli(key, out, request):
    cfg=MODELS[key];cmd=request['command'];private=private_dir(key)
    env=os.environ.copy()
    for name in list(env):
        if name.startswith(('CODEX_','CLAUDE_')) or name in ['OPENAI_API_KEY','ANTHROPIC_API_KEY','OPENROUTER_API_KEY']:env.pop(name,None)
    env['PYTHONIOENCODING']='utf-8'
    credential=None
    if cfg['backend']=='codex':
        flags=[]
        for i,arg in enumerate(cmd):
            if arg=='-c':flags.extend(['-c',cmd[i+1]])
        original=Path(os.environ.get('CODEX_HOME',str(Path.home()/'.codex')))
        env['CODEX_HOME']=str(private)
        pre=subprocess.run([cmd[0],'-C',str(private/'empty')]+flags+['debug','prompt-input','PREFLIGHT; NO MODEL CALL.'],
                           env=env,capture_output=True,text=True,encoding='utf-8',timeout=60)
        (out/'preflight.json').write_text(pre.stdout,encoding='utf-8')
        assert pre.returncode==0,pre.stderr
        assert not any(s in pre.stdout for s in ['<skills_instructions>','### Available skills','# AGENTS.md instructions'])
        credential=private/'auth.json';assert not credential.exists(),'Unresolved prior private credential'
        shutil.copyfile(original/'auth.json',credential)
    if cfg['backend']=='claude':env['CLAUDE_CODE_MAX_OUTPUT_TOKENS']=str(request.get('output_token_cap',64000))
    events=[];started=utc();tick=time.monotonic()
    try:
        with (out/'events.jsonl').open('w',encoding='utf-8') as log,(out/'stderr.txt').open('w',encoding='utf-8') as err:
            child=subprocess.Popen(cmd,cwd=private/'empty',env=env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=err,text=True,encoding='utf-8',bufsize=1)
            dump(out/'status.json',{'status':'running','started_utc':started,'pid':child.pid,'model':cfg['model']})
            if cfg['backend']!='muse':child.stdin.write(request['prompt'])
            child.stdin.close()
            for line in child.stdout:
                log.write(line);log.flush()
                try:events.append(json.loads(line))
                except json.JSONDecodeError:pass
            code=child.wait()
    finally:
        if credential is not None and credential.exists():credential.unlink()
    parsed=parse_cli(cfg['backend'],events)
    return dict(parsed,events=events,exit_code=code,started_utc=started,finished_utc=utc(),wall_seconds=time.monotonic()-tick)

def openrouter_body(key,prompt):
    cfg=MODELS[key]
    body={'model':cfg['model'] if key=='nemotron' else cfg['canonical'],'messages':[{'role':'user','content':prompt}],
          'temperature':0,'max_tokens':MAX_OUTPUT_TOKENS,'reasoning':{'enabled':True,'exclude':False},
          'response_format':{'type':'json_schema','json_schema':{'name':'mlp_weights','strict':True,'schema':SCHEMA}},
          'tool_choice':'none','plugins':[]}
    if key=='gemini':body['reasoning']={'effort':'high','exclude':False}
    return body

def run_openrouter(key,out,request):
    cfg=MODELS[key];headers={'Authorization':'Bearer '+os.environ['OPENROUTER_API_KEY']}
    started=utc();tick=time.monotonic();events=[];parts=[];reasons=[];usage={};tools=[];completed=False;error=None
    if cfg['backend']=='batch':
        submit=out/'batch_submission.json'
        if submit.exists():batch=json.loads(submit.read_text(encoding='utf-8'))
        else:
            response=requests.post('https://openrouter.ai/api/v1/batches',headers=headers,json=request['body'],timeout=(30,120))
            batch=response.json();dump(submit,batch)
            assert response.ok,(response.status_code,batch)
        assert batch.get('id'),batch
        dump(out/'status.json',{'status':'batch_pending','id':batch['id'],'started_utc':started})
        with (out/'batch_polls.jsonl').open('a',encoding='utf-8') as log:
            while batch.get('status') not in ['completed','failed','cancelled','expired']:
                time.sleep(30)
                response=requests.get('https://openrouter.ai/api/v1/batches/'+batch['id'],headers=headers,timeout=30)
                response.raise_for_status();batch=response.json()
                log.write(json.dumps(batch,ensure_ascii=False)+'\n');log.flush();dump(out/'batch_latest.json',batch)
        result=(batch.get('results') or [{}])[0]
        data=(result.get('response') or {}).get('body',{})
        dump(out/'batch_result.json',batch);events=[data];usage=data.get('usage',{})
        choice=(data.get('choices') or [{}])[0];message=choice.get('message',{})
        parts=[message.get('content') or ''];reasons=[message.get('reasoning') or message.get('reasoning_content') or '']
        tools=message.get('tool_calls') or [];completed=batch.get('status')=='completed' and choice.get('finish_reason')=='stop'
        error=result.get('error') or batch.get('error')
    else:
        body=request['body']
        with (out/'events.jsonl').open('w',encoding='utf-8') as log,(out/'reasoning.txt').open('w',encoding='utf-8') as reasonlog, \
             (out/'weights_output.json').open('w',encoding='utf-8') as textlog:
            with requests.post('https://openrouter.ai/api/v1/chat/completions',headers=headers,json=body,stream=True,timeout=(30,300)) as response:
                if not response.ok: error={'status':response.status_code,'body':response.text}
                else:
                    for line in response.iter_lines(chunk_size=1,decode_unicode=True):
                        if not line or not line.startswith('data:'):continue
                        if line[5:].strip()=='[DONE]':completed=True;break
                        e=json.loads(line[5:]);events.append(e);log.write(json.dumps(e,ensure_ascii=False)+'\n');log.flush()
                        if e.get('usage'):usage=e['usage']
                        if e.get('error'):error=e['error']
                        for choice in e.get('choices',[]):
                            delta=choice.get('delta',{})
                            content=delta.get('content') or '';reason=delta.get('reasoning') or delta.get('reasoning_content') or ''
                            if not reason:reason=''.join(x.get('text',x.get('summary','')) for x in delta.get('reasoning_details',[]) if x.get('type') in ['reasoning.text','reasoning.summary'])
                            parts.append(content);reasons.append(reason);textlog.write(content);textlog.flush();reasonlog.write(reason);reasonlog.flush()
                            if delta.get('tool_calls'):tools.append(delta['tool_calls'])
                            if choice.get('finish_reason') and choice['finish_reason']!='stop':error={'finish_reason':choice['finish_reason']}
    native_cost=(batch.get('usage') or {}).get('cost') if cfg['backend']=='batch' else usage.get('cost')
    return {'answer':''.join(parts),'reasoning':''.join(reasons),'usage':usage,'completed':completed and not error,
            'tool_events':tools,'events':events,'error':error,'started_utc':started,'finished_utc':utc(),
            'wall_seconds':time.monotonic()-tick,'cost_usd':native_cost}

def collect_one(key):
    cfg=MODELS[key];out=DATA/'responses'/key;out.mkdir(parents=True,exist_ok=True)
    prompt=(DATA/'prompt.txt').read_text(encoding='utf-8')
    assert sha(prompt)==json.loads((DATA/'manifest.json').read_text(encoding='utf-8'))['prompt_sha256']
    request={'config':cfg,'prompt':prompt,'schema':SCHEMA,'tools_enabled':[],'protocol':'digits-v1'}
    if cfg['backend'] in ['codex','claude','muse']:request['command']=cli_command(key,out)
    else:
        meta=requests.get('https://openrouter.ai/api/v1/models',timeout=30).json()['data']
        model=next(m for m in meta if m['id']==cfg['model']);assert model['canonical_slug']==cfg['canonical'];dump(out/'model_metadata.json',model)
        body=openrouter_body(key,prompt)
        if key=='nemotron':
            assert all(float(model['pricing'][k])==0 for k in ['prompt','completion'])
            body.update(provider={'only':['nvidia'],'allow_fallbacks':False,'max_price':{'prompt':0,'completion':0,'request':0}},
                        stream=True,stream_options={'include_usage':True})
        else:body={'endpoint':'/v1/chat/completions','model':cfg['canonical'],
                   'provider':{'only':['google-vertex'],'allow_fallbacks':False},'completion_window':'24h',
                   'requests':[{'custom_id':'digits-v1-gemini','body':body}]}
        request['body']=body
    keyhash=sha(json.dumps(request,sort_keys=True));cache=out/'response.json'
    if cache.exists():
        record=json.loads(cache.read_text(encoding='utf-8'));assert record['cache_key']==keyhash
        validate_and_score(key,record);return
    assert not (out/'started.json').exists(),'Prior request unresolved; inspect logs instead of regenerating'
    dump(out/'request.json',request);dump(out/'started.json',{'started_utc':utc(),'cache_key':keyhash})
    # Health check before a fresh call. Keys are never saved.
    try:requests.get(os.environ.get('LANGFUSE_HOST','http://localhost:3300').rstrip('/')+'/api/public/health',timeout=2)
    except requests.RequestException:pass
    print('START '+key,flush=True)
    try:
        record=run_cli(key,out,request) if cfg['backend'] in ['codex','claude','muse'] else run_openrouter(key,out,request)
    except Exception as exc:
        dump(out/'failure.json',{'exception':type(exc).__name__,'message':str(exc),'at_utc':utc()});raise
    record.update(cache_key=keyhash,request=request);dump(cache,record)
    (out/'weights_output.json').write_text(record.get('answer',''),encoding='utf-8')
    (out/'reasoning.txt').write_text(record.get('reasoning',''),encoding='utf-8')
    trace_call(keyhash,cfg,request,record)
    validate_and_score(key,record)
    print('FINISH '+key,flush=True)

def load_submission(text):
    payload=json.loads(text,parse_constant=lambda x: (_ for _ in ()).throw(ValueError('Non-finite JSON value')))
    assert isinstance(payload,dict) and set(payload)=={'weights'},'Wrong JSON keys'
    values=payload['weights'];assert isinstance(values,list),'Weights must be a JSON array'
    assert len(values)==N_PARAMS,f'Expected {N_PARAMS} weights; received {len(values)}'
    assert all(type(v) in [int,float] for v in values),'Non-numeric weight'
    flat=np.asarray(values,dtype=np.float64);assert np.isfinite(flat).all(),'Non-finite weight'
    arrays={};offset=0
    for name,shape in SHAPES.items():
        size=int(np.prod(shape));arrays[name]=flat[offset:offset+size].reshape(shape);offset+=size
    assert np.array_equal(np.concatenate([a.ravel() for a in arrays.values()]),flat)
    return arrays,flat

def forward(raw, arrays):
    x=np.asarray(raw,dtype=np.float64)/16.0
    h1=np.maximum(x@arrays['W1'].T+arrays['b1'],0)
    h2=np.maximum(h1@arrays['W2'].T+arrays['b2'],0)
    return h2@arrays['W3'].T+arrays['b3']

def wilson(k,n):
    z=1.959963984540054;p=k/n;den=1+z*z/n
    centre=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [centre-half,centre+half]

def validate_and_score(key,record,out_override=None):
    out=Path(out_override) if out_override else DATA/'responses'/key
    result={'model':MODELS[key]['model'],'model_key':key,'format_failure':None,'failure_category':None,'tool_events':len(record.get('tool_events',[])),
            'tester_changed_weights':False,'training_steps':0,'parameters':N_PARAMS,
            'usage':record.get('usage'),'cost_usd':record.get('cost_usd'),'wall_seconds':record.get('wall_seconds'),
            'reasoning_characters':len(record.get('reasoning',''))}
    try:
        assert record.get('completed') and record.get('exit_code',0)==0,'Incomplete or failed model response'
        assert not record.get('tool_events'),'Tool protocol failure'
        for cat in record.get('catalogues',[]):
            assert not any(cat.get(k) for k in ['tools','mcp_servers','skills']),'Nonempty tool/customization catalogue'
            assert all(p.get('path')=='builtin' for p in cat.get('plugins',[])),'Unexpected non-builtin plugin'
        arrays,flat=load_submission(record['answer']);np.savez_compressed(out/'candidate.npz',**arrays)
        with np.load(out/'candidate.npz') as saved:assert all(np.array_equal(saved[k],v) for k,v in arrays.items())
        result['weights']={'zeros':int(np.sum(flat==0)),'max_abs':float(np.abs(flat).max()),'mean':float(flat.mean()),'rms':float(np.sqrt(np.mean(flat**2)))}
        with np.load(DATA/'dataset.npz') as data:
            predictions={}
            for split in ['visible','hidden_train','test']:
                rows=data[split];scores=forward(rows[:,:64],arrays);assert np.isfinite(scores).all(),'Non-finite network scores'
                pred=scores.argmax(axis=1);gold=rows[:,-1];correct=pred==gold
                confusion=np.zeros((10,10),dtype=int);np.add.at(confusion,(gold,pred),1)
                result[split]={'n':len(gold),'correct':int(correct.sum()),'accuracy':float(correct.mean()),
                               'wilson_95':wilson(int(correct.sum()),len(gold)),'confusion':confusion.tolist(),
                               'per_class_accuracy':(np.diag(confusion)/confusion.sum(axis=1)).tolist()}
                predictions[split+'_pred']=pred;predictions[split+'_scores']=scores
            np.savez_compressed(out/'predictions.npz',**predictions)
    except (AssertionError,ValueError,TypeError,OverflowError) as exc:
        result['format_failure']=str(exc) or type(exc).__name__
        category='unreadable_format'
        terminal=[ev for ev in record.get('events',[]) if ev.get('type')=='result']
        finish=[c.get('finish_reason') for ev in record.get('events',[]) for c in ev.get('choices',[]) if c.get('finish_reason')]
        if terminal and terminal[-1].get('safety_stops') and record.get('answer','').strip():category='refusal'
        elif terminal and terminal[-1].get('subtype')=='error_max_budget_usd':category='budget_or_output_stop'
        elif 'length' in finish or (record.get('error') or {}).get('finish_reason')=='length':category='output_limit'
        elif record.get('error'):category='provider_or_transport_failure'
        elif not record.get('completed'):category='incomplete_response'
        elif result['format_failure'].startswith('Expected 6570 weights'):category='wrong_weight_count'
        result['failure_category']=category
    dump(out/'results.json',result)
    dump(out/'status.json',{'status':'scored' if result['format_failure'] is None else 'unreadable','finished_utc':utc(),'format_failure':result['format_failure']})
    return result

# Reproduction is offline; collection helpers are defined but never called.
MAX_NEW_CALLS = 0
ALLOW_NEW_REQUESTS = False
"""Cache-only figures and measurements. No teacher calls and no candidate fitting."""
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA = ROOT/'data'
FIGURES = ROOT/'figures'
MODEL_LABELS = {'astra':'Astra','sol':'Sol','opus':'Opus 5.5','muse':'Muse Wonder','nemotron':'Nemotron','gemini':'Gemini batch',
                'sonnet':'Sonnet 5.5','haiku':'Haiku 5.5','fable':'Fable 5.1','deepseek':'DeepSeek V4.1 Flash','glm':'GLM-5.3','mimo':'MiMo-V2.6-Pro'}
OPEN_KEYS = ['deepseek','glm','mimo']
FAMILY_LABELS = {'fable':'Fable 5.1','opus':'Opus 5.5 high','sonnet':'Sonnet 5.5','haiku':'Haiku 5.5'}
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':160,'font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'bold','axes.labelsize':10,
                     'figure.facecolor':'white','axes.facecolor':'white'})

def save_json(path,obj):
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')

def live_reasoning_logs():
    """Only text the external provider actually emitted. Never infer hidden reasoning."""
    folders=[(key,DATA/'responses'/key) for key in MODEL_LABELS]+[(key,DATA/'claude_family'/key) for key in FAMILY_LABELS]
    for key,folder in folders:
        events_path=folder/'events.jsonl'
        if not events_path.exists():continue
        events=[]
        for line in events_path.read_text(encoding='utf-8').splitlines():
            try:events.append(json.loads(line))
            except ValueError:pass
        if key in FAMILY_LABELS:
            chunks=[e.get('event',{}).get('delta',{}).get('thinking','') for e in events
                    if e.get('event',{}).get('delta',{}).get('type')=='thinking_delta']
            full=[b.get('thinking','') for e in events if e.get('type')=='assistant'
                  for b in e.get('message',{}).get('content',[]) if b.get('type')=='thinking' and b.get('thinking','').strip()]
            (folder/'reasoning.txt').write_text('\n'.join(full) if full else ''.join(chunks),encoding='utf-8')
        elif key=='muse':
            chunks=[]
            for e in events:
                p=e.get('payload',{});kind=e.get('payload_type','')
                if 'reasoning' in kind or 'thinking' in kind:
                    value=p.get('text') or p.get('delta')
                    if isinstance(value,str):chunks.append(value)
            if chunks:(folder/'reasoning.txt').write_text(''.join(chunks),encoding='utf-8')

def collect_results():
    results=[]
    for key in ['astra','sol','opus','muse','nemotron','gemini']:
        label=MODEL_LABELS[key];folder=DATA/'responses'/key;path=folder/'results.json'
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=label
        result['candidate_folder']=str(folder.relative_to(DATA))
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8')) if rp.exists() else 0
        results.append(result)
    for key,label in FAMILY_LABELS.items():
        folder=DATA/'claude_family'/key;path=folder/'results.json'
        if not folder.exists():continue
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=label;result['family']=True
        result['candidate_folder']=str(folder.relative_to(DATA))
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8').strip()) if rp.exists() else 0
        results.append(result)
    for key in OPEN_KEYS:
        folder=DATA/'open_models'/key;path=folder/'results.json'
        if not folder.exists():continue
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=MODEL_LABELS[key];result['open']=True
        result['candidate_folder']=str(folder.relative_to(DATA));result['outer_requests']=len(list(folder.glob('call-*/response.json')))
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8').strip()) if rp.exists() else 0
        results.append(result)
    for recovery_key in ['opus','gemini','gemini-low']:
        key=recovery_key.split('-')[0];folder=DATA/'recovery'/recovery_key
        if not folder.exists():continue
        path=folder/'results.json'
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=MODEL_LABELS[key]+(' continuation' if key=='opus' else (' low effort' if recovery_key=='gemini-low' else ' schema retry'))
        result['candidate_folder']=str(folder.relative_to(DATA));result['recovery']=True
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8').strip()) if rp.exists() else 0
        results.append(result)
    indexed_root=DATA/'indexed'
    if indexed_root.exists():
        order=['gemini','nemotron','fable','opus','sonnet','haiku']
        for folder in sorted(indexed_root.iterdir(),key=lambda f:order.index(f.name) if f.name in order else len(order)):
            if not folder.is_dir() or folder.name not in MODEL_LABELS:continue
            key=folder.name;path=folder/'results.json'
            result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
            result['label']=MODEL_LABELS[key]+(' high' if key=='opus' else '')+' numbered entries';result['indexed']=True
            result['candidate_folder']=str(folder.relative_to(DATA));result['outer_requests']=len(list(folder.glob('turn-*/response.json')))
            result['reasoning_characters']=len((folder/'reasoning.txt').read_text(encoding='utf-8').strip()) if (folder/'reasoning.txt').exists() else 0
            results.append(result)
    return results

def nearest_neighbour_baseline(dataset):
    visible=dataset['visible'];x=visible[:,:64].astype(float)/16;labels=visible[:,-1]
    metrics={};predictions={}
    for split in ['hidden_train','test']:
        z=dataset[split][:,:64].astype(float)/16;pred=[]
        for i in range(0,len(z),128):
            batch=z[i:i+128];distance=(batch**2).sum(1)[:,None]+(x**2).sum(1)[None,:]-2*batch@x.T
            pred.extend(labels[np.argmin(distance,axis=1)])
        pred=np.asarray(pred);gold=dataset[split][:,-1];predictions[split]=pred
        metrics[split]={'n':len(gold),'correct':int(np.sum(gold==pred)),'accuracy':float(np.mean(gold==pred))}
    np.savez_compressed(DATA/'nearest_neighbour_predictions.npz',**predictions)
    save_json(DATA/'nearest_neighbour_baseline.json',metrics)
    return metrics

def verify_legacy_mlps():
    results=[];legacy=DATA/'legacy'
    points=np.load(legacy/'mlp17/data/evaluation_points.npz')
    for name in ['astra','nemotron-no-reasoning']:
        folder=legacy/'mlp17/data'/name
        if not (folder/'classification.npz').exists():continue
        row={'model':name,'parameters':17}
        for task in ['classification','regression']:
            with np.load(folder/(task+'.npz')) as w:
                x=points[task+'_x'];scores=np.maximum(x@w['W1'].T+w['b1'],0)@w['W2']+w['b2'][0]
            gold=points[task+'_y'];row[task+'_n']=len(gold)
            if task=='classification':row['correct']=int(np.sum((scores>0)==gold));row['accuracy']=float(np.mean((scores>0)==gold))
            else:row['regression_mae']=float(np.mean(np.abs(scores-gold)))
        results.append(row)
    points.close()
    with np.load(legacy/'rings388/data/test_points.npz') as points:
        for name in ['astra','nemotron-free']:
            with np.load(legacy/'rings388/data'/name/'candidate.npz') as w:
                x=points['x'];h1=np.maximum(x@w['W1'].T+w['b1'],0);h2=np.maximum(h1@w['W2'].T+w['b2'],0)
                pred=(h2@w['W3'].T+w['b3']).argmax(axis=1);gold=points['y']
            expected_path=legacy/'rings388/data'/name/'results.json'
            if not expected_path.exists():expected_path=legacy/'rings388/results.json'
            expected=json.loads(expected_path.read_text())
            correct=int(np.sum(pred==gold));assert correct==expected['correct']
            results.append({'model':name,'parameters':388,'classification_n':len(gold),'correct':correct,'accuracy':float(np.mean(pred==gold))})
    save_json(DATA/'legacy_replay.json',results)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,size,chance,title in zip(axes,[17,388],[50,25],['Known opposite-sign rule','Known four-ring rule']):
        subset=[r for r in results if r['parameters']==size]
        ax.bar([r['model'].replace('nemotron-no-reasoning','Nemotron\nreasoning off').replace('nemotron-free','Nemotron free').replace('astra','Astra') for r in subset],
               [100*r['accuracy'] for r in subset],color=['#286f9e','#7f9a61'])
        ax.axhline(chance,color='#777',linestyle='--');ax.set(ylim=(0,110),ylabel='Cases classified correctly (%)',title=f'{size} parameters: {title}')
        for j,r in enumerate(subset):ax.text(j,100*r['accuracy']+3,f"{r['correct']:,}/{r['classification_n']:,}",ha='center',fontsize=9)
    fig.tight_layout();fig.savefig(FIGURES/'legacy_accuracy.png',bbox_inches='tight');plt.close(fig)
    return results

GROUPS = ['First request to each model','Claude models at high effort','Open-weight models, numbered from the start','Retries after a failed first reply']
BLUE,GREY,BROWN,INK = '#286f9e','#9aa3ad','#b7752c','#172e40'
AXIS_CAP = 200000  # Haiku's flat reply (256,483 tokens) is drawn cut at this axis limit and labelled.

def workflow_group(r):
    if r.get('family'):return GROUPS[1]
    if r.get('open'):return GROUPS[2]
    if r.get('recovery') or r.get('indexed'):return GROUPS[3]
    return GROUPS[0]

def failure_note(r):
    """Short reason for a submission that produced no scored network."""
    if r.get('pending'):return 'Still running'
    cat=r.get('failure_category');text=r.get('format_failure') or ''
    if r['model_key']=='gemini' and workflow_group(r)==GROUPS[0]:return 'No weights: provider rejected the output schema'
    if r['model_key']=='haiku' and r.get('family'):return 'No weights: two runs of zeros hit the output cap, then a note'
    if cat=='declined':return 'No weights: declined to write untested weights'
    if cat=='wrong_weight_count':
        m=re.search(r'received (\d+)',text)
        return f'Wrong count: {int(m.group(1)):,} of 6,570 values' if m else 'Wrong number of values'
    return {'budget_or_output_stop':'No weights: reasoning used the whole output allowance',
            'refusal':'No weights: refusal','output_limit':'No weights: output limit reached during reasoning',
            'provider_unavailable':'No weights: three provider timeouts','provider_or_transport_failure':'No weights: provider error',
            'incomplete_response':'No weights: incomplete response','unreadable_format':'Reply did not parse as the JSON contract'}.get(cat,'No usable network')

def ordered_rows(results):
    return [r for g in GROUPS for r in results if workflow_group(r)==g]

def usage_row(r):
    """Native token counters, wall time and reported cost for one submission, without estimating missing values."""
    folder=DATA/r['candidate_folder'];p=folder/'response.json'
    if r.get('indexed') or r.get('open'):p=folder/('assembled_response.json' if (folder/'assembled_response.json').exists() else 'workflow_summary.json')
    native=json.loads(p.read_text(encoding='utf-8')) if p.exists() else {}
    usage=native.get('usage') or {};backend=((native.get('request') or {}).get('config') or {}).get('backend')
    input_tokens=usage.get('input_tokens',usage.get('prompt_tokens'));output_tokens=usage.get('output_tokens',usage.get('completion_tokens'))
    reasoning=usage.get('reasoning_output_tokens',(usage.get('completion_tokens_details') or {}).get('reasoning_tokens'))
    if usage.get('per_turn'):
        reported=[u for u in usage['per_turn'] if u and (u.get('prompt_tokens') is not None or u.get('output_tokens') is not None)]
        def turn_input(u):
            if u.get('prompt_tokens') is not None:return u['prompt_tokens']
            return sum(u.get(k) or 0 for k in ['input_tokens','cache_creation_input_tokens','cache_read_input_tokens'])
        def turn_reasoning(u):
            return (u.get('completion_tokens_details') or {}).get('reasoning_tokens') or (u.get('output_tokens_details') or {}).get('thinking_tokens') or 0
        input_tokens=sum(turn_input(u) for u in reported) if reported else None
        output_tokens=sum(u.get('completion_tokens',u.get('output_tokens',0)) for u in reported) if reported else None
        reasoning=sum(turn_reasoning(u) for u in reported) if reported else None
    if r['model_key']=='muse' and usage.get('models'):
        u=usage['models'][0];input_tokens=u.get('input_tokens');output_tokens=u.get('output_tokens');reasoning=u.get('reasoning_tokens')
    if backend=='claude' and not usage.get('per_turn'):
        # Claude reports cached prompt tokens separately from the uncached remainder.
        input_tokens=sum(usage.get(k) or 0 for k in ['input_tokens','cache_creation_input_tokens','cache_read_input_tokens']) if usage else None
        reasoning=(usage.get('output_tokens_details') or {}).get('thinking_tokens')
    cost=r.get('cost_usd') if r.get('cost_usd') is not None else native.get('cost_usd')
    return {'label':r['label'],'model_key':r['model_key'],'group':workflow_group(r),'usable':r.get('format_failure') is None and not r.get('pending'),
            'pending':bool(r.get('pending')),'failure_category':r.get('failure_category'),
            'input_tokens':input_tokens,'output_tokens':output_tokens,'reasoning_tokens':reasoning,
            'wall_seconds':native.get('wall_seconds') if p.exists() else None,'cost_usd':cost,'evidence':str(p.relative_to(DATA)) if p.exists() else None}

def _group_axis(ax,rows,headers):
    labels=[];weights=[]
    for kind,text in rows:
        labels.append(text);weights.append('bold' if kind=='header' else 'normal')
    ax.set_yticks(range(len(rows)));ax.set_yticklabels(labels);ax.set_ylim(len(rows)-.3,-.7)
    for tick,weight in zip(ax.get_yticklabels(),weights):tick.set_fontweight(weight)
    ax.tick_params(axis='y',length=0)

def accuracy_figure(results,valid,baseline):
    rows=[];items=[]
    for g in GROUPS:
        members=[r for r in results if workflow_group(r)==g]
        if not members:continue
        rows.append(('header',g));items.append(None)
        for r in members:rows.append(('row','   '+r['label']));items.append(r)
    rows.append(('header','Reference that stores the sample'));items.append(None)
    rows.append(('row','   1-nearest neighbour'));items.append('reference')
    fig,ax=plt.subplots(figsize=(10.5,0.34*len(rows)+1.5))
    for y,item in enumerate(items):
        if item is None:continue
        if item=='reference':
            h=100*baseline['test']['accuracy'];ax.barh(y,h,color=BROWN,height=.62)
            ax.text(h+1,y,f"{h:.1f}%  ({baseline['test']['correct']:,}/1,797)",va='center',fontsize=8.5)
        elif item in valid:
            h=100*item['test']['accuracy'];lo,hi=np.asarray(item['test']['wilson_95'])*100
            ax.barh(y,h,color=BLUE,height=.62);ax.errorbar(h,y,xerr=[[h-lo],[hi-h]],fmt='none',ecolor=INK,capsize=2.5,lw=1)
            ax.text(hi+1,y,f"{h:.1f}%  ({item['test']['correct']:,}/1,797)",va='center',fontsize=8.5)
        else:
            ax.plot(0.6,y,marker='x',color=GREY,ms=6,mew=1.5)
            ax.text(2.2,y,failure_note(item),va='center',fontsize=8.5,color='#4b535c',style='italic')
    ax.axvline(10,color='#777777',linestyle='--',lw=1);ax.text(10.6,0,'random guessing, 10%',fontsize=8,color='#555555',va='center')
    best=max(valid,key=lambda r:r['test']['accuracy']) if valid else None
    title=f"{len(valid)} of {len(results)} submissions produced a network that could be scored"
    if best:title+=f"; {best['label']} scored highest ({100*best['test']['accuracy']:.1f}%)"
    ax.set(xlim=(0,112),xlabel='Official test images classified correctly (%), 1,797 images');ax.set_title(title,fontsize=11)
    _group_axis(ax,rows,GROUPS)
    ax.annotate('Bars: one fixed network each, with Wilson 95% intervals over test images. A cross marks a submission with no network to score;\nit is not an accuracy of zero. The reference copies the label of the most similar visible image and is not a directly written network.',
                xy=(0,0),xycoords='axes fraction',xytext=(0,-34),textcoords='offset points',fontsize=7.8,color='#444444',va='top')
    fig.tight_layout();fig.savefig(FIGURES/'test_accuracy.png',bbox_inches='tight');plt.close(fig)

def usage_figure(results,valid):
    ordered=ordered_rows(results);usage=[usage_row(r) for r in ordered]
    rows=[];items=[]
    for g in GROUPS:
        members=[u for u in usage if u['group']==g]
        if not members:continue
        rows.append(('header',g));items.append(None)
        for u in members:rows.append(('row','   '+u['label']));items.append(u)
    fig,ax=plt.subplots(figsize=(10.5,0.34*len(rows)+1.5))
    for y,u in enumerate(items):
        if u is None:continue
        out=u['output_tokens'];rt=u['reasoning_tokens']
        if out is None:
            text=('Still running' if u['pending'] else 'Provider error before any output' if u['failure_category'] in ['provider_unavailable','provider_or_transport_failure']
                  else 'No token count reported')
            ax.text(1000,y,text,va='center',fontsize=8.5,color='#4b535c',style='italic');continue
        rt=min(rt or 0,out);answer=out-rt;cut=out>AXIS_CAP
        colour=BLUE if u['usable'] else GREY
        ax.barh(y,min(rt,AXIS_CAP),color=colour,alpha=.45,height=.62,hatch='///' if not u['usable'] else None,edgecolor='white')
        ax.barh(y,min(answer,AXIS_CAP-min(rt,AXIS_CAP)),left=min(rt,AXIS_CAP),color=colour,height=.62)
        note=f"{out:,} output tokens"+(f", {rt:,} reasoning" if rt else '')+('' if u['usable'] else ' - no network')+(' (bar cut)' if cut else '')
        if cut:ax.text(rt+3000,y,note,va='center',ha='left',fontsize=8.2,color='white')
        else:ax.text(out+1500,y,note,va='center',fontsize=8.2)
    ax.set_xlabel('Output tokens per submission (light: reported reasoning; solid: visible reply)')
    ax.set_xlim(0,AXIS_CAP);ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v,_:f'{int(v):,}'))
    known=[u for u in usage if u['output_tokens'] is not None]
    scored=[u['output_tokens'] for u in known if u['usable']]
    limited=[u for u in known if u['failure_category'] in ['budget_or_output_stop','output_limit']]
    title=f"Networks that could be scored used {min(scored):,} to {max(scored):,} output tokens" if scored else 'Output tokens per submission'
    if limited:title+=f"; {len(limited)} failures stopped at an output limit"
    ax.set_title(title,fontsize=11)
    _group_axis(ax,rows,GROUPS)
    fig.tight_layout();fig.savefig(FIGURES/'output_tokens.png',bbox_inches='tight');plt.close(fig)
    return usage

def make_figures():
    FIGURES.mkdir(exist_ok=True);live_reasoning_logs();indexed_workflow_metadata();reported_billing();verify_indexed_artifacts()
    with np.load(DATA/'dataset.npz') as saved:dataset={k:saved[k] for k in saved.files}
    results=collect_results();valid=[r for r in results if not r.get('pending') and r.get('format_failure') is None]
    legacy=verify_legacy_mlps()
    baseline=nearest_neighbour_baseline(dataset)
    parameter_rows=[]
    for r in valid:
        with np.load(DATA/r['candidate_folder']/'candidate.npz') as weights:
            parameter_rows.append({'label':r['label'],'zeros':r['weights']['zeros'],'max_abs':r['weights']['max_abs'],
                'h1_nonzero_rows':int(np.sum(np.any(weights['W1']!=0,axis=1)|(weights['b1']!=0))),
                'h2_nonzero_rows':int(np.sum(np.any(weights['W2']!=0,axis=1)|(weights['b2']!=0))),
                'nonzero_by_tensor':{k:int(np.count_nonzero(weights[k])) for k in weights.files}})
    save_json(DATA/'parameter_use.json',parameter_rows)
    astra_path=DATA/'responses/astra/candidate.npz';sol_path=DATA/'responses/sol/candidate.npz'
    if astra_path.exists() and sol_path.exists():
        structure={'analysis':'Post-hoc interpretation, not teacher reasoning text','identical_downstream':{},'max_identity_error':{}}
        with np.load(astra_path) as first,np.load(sol_path) as second:
            structure['identical_downstream']={k:bool(np.array_equal(first[k],second[k])) for k in ['W2','b2','W3','b3']}
            for name,w in [('Astra',first),('Sol',second)]:
                x=dataset['test'][:,:64].astype(float)/16;h1=np.maximum(x@w['W1'].T+w['b1'],0)
                blocks=h1[:,:40].reshape(-1,10,4)
                interpreted=np.maximum(blocks[:,:,:2].sum(axis=2),blocks[:,:,2:].sum(axis=2))
                h2=np.maximum(h1@w['W2'].T+w['b2'],0);actual=h2@w['W3'].T+w['b3']
                structure['max_identity_error'][name]=float(np.max(np.abs(actual-interpreted)))
        x=dataset['test'][:,:64].astype(float)/16
        claude_paths={'Opus 5.5 high numbered entries':DATA/'indexed/opus/candidate.npz','Sonnet 5.5 numbered entries':DATA/'indexed/sonnet/candidate.npz'}
        structure['template_matching']={}
        for name,path in claude_paths.items():
            if not path.exists():continue
            with np.load(path) as w:
                h1=np.maximum(x@w['W1'].T+w['b1'],0);h2=np.maximum(h1@w['W2'].T+w['b2'],0);actual=h2@w['W3'].T+w['b3']
                if name.startswith('Opus'):
                    # Each digit score is the best of its first-layer template units (two templates for 1, 7 and 9).
                    units={c:[int(u) for u in np.flatnonzero(w['W2'][np.flatnonzero(w['W3'][c])].max(axis=0)>0)] for c in range(10)}
                    interpreted=np.stack([h1[:,units[c]].max(axis=1) for c in range(10)],axis=1)
                    entry={'templates_per_digit':{str(c):len(units[c]) for c in range(10)},'template_units':int(np.sum(np.any(w['W1']!=0,axis=1))),
                           'smallest_template_preactivation':float((x@w['W1'].T+w['b1'])[:,np.any(w['W1']!=0,axis=1)].min())}
                else:
                    # W1 is the identity, so each digit score is one template dot product plus a bias.
                    interpreted=x@w['W2'][:10].T+w['b2'][:10]
                    entry={'first_layer_is_identity':bool(np.array_equal(w['W1'],np.eye(64))),'template_units':int(np.sum(np.any(w['W2']!=0,axis=1)))}
                entry['max_difference_from_full_network']=float(np.max(np.abs(actual-interpreted)))
                structure['template_matching'][name]=entry
        save_json(DATA/'program_structure.json',structure)
    if parameter_rows:
        fig,ax=plt.subplots(figsize=(10,4));bottom=np.zeros(len(parameter_rows));colours=['#286f9e','#7f9a61','#b7752c']
        for names,label,colour in zip([['W1','b1'],['W2','b2'],['W3','b3']],['Input layer','Second layer','Output layer'],colours):
            counts=np.array([sum(r['nonzero_by_tensor'][k] for k in names) for r in parameter_rows]);ax.bar(range(len(counts)),counts,bottom=bottom,color=colour,label=label);bottom+=counts
        for i,total in enumerate(bottom):ax.text(i,total+60,f'{int(total):,}',ha='center',fontsize=8.5)
        ax.axhline(6570,color='#777777',linestyle='--',lw=1);ax.text(len(parameter_rows)-.5,6570+60,'all 6,570 values',ha='right',fontsize=8,color='#555555')
        ax.set(xticks=range(len(parameter_rows)),ylim=(0,7300),ylabel='Nonzero values (of 6,570 supplied)',title='The networks that run use between {:,} and {:,} nonzero values'.format(int(bottom.min()),int(bottom.max())))
        ax.set_xticklabels([r['label'] for r in parameter_rows],rotation=20,ha='right')
        ax.legend(fontsize=8);fig.tight_layout();fig.savefig(FIGURES/'parameter_use.png',bbox_inches='tight');plt.close(fig)
    fig,axes=plt.subplots(2,5,figsize=(10,4.5))
    for digit,ax in enumerate(axes.flat):
        i=int(np.flatnonzero(dataset['visible'][:,-1]==digit)[0]);row=dataset['visible'][i]
        ax.imshow(row[:64].reshape(8,8),cmap='Greys',vmin=0,vmax=16,interpolation='nearest')
        ax.set_title(f'Digit {digit}: train-{int(dataset["visible_indices"][i]):04d}',fontsize=9);ax.axis('off')
    fig.suptitle('The visible sample contains ten handwritten digit classes',fontsize=13)
    fig.tight_layout();fig.savefig(FIGURES/'visible_digits.png',bbox_inches='tight');plt.close(fig)
    accuracy_figure(results,valid,baseline)
    usage=usage_figure(results,valid)
    save_json(DATA/'usage_summary.json',usage)
    example_ids=[];selection=[]
    def add(i,why):
        if int(i) not in example_ids:example_ids.append(int(i));selection.append(why)
    add(0,'First official test item, fixed ordinary case')
    if valid:
        lead=sorted(valid,key=lambda r:(-r['test']['accuracy'],r['model_key']))[0]
        with np.load(DATA/lead['candidate_folder']/'predictions.npz') as p:pred=p['test_pred'];scores=p['test_scores']
        gold=dataset['test'][:,-1]
        for mask,why in [(pred==gold,'First correct case for highest observed score'),(pred!=gold,'First error for highest observed score')]:
            indices=np.flatnonzero(mask)
            if len(indices):add(indices[0],why)
        sorted_scores=np.sort(scores,axis=1);add(np.argmin(sorted_scores[:,-1]-sorted_scores[:,-2]),'Smallest uncalibrated top-two score margin')
        if len(valid)>1:
            with np.load(DATA/valid[0]['candidate_folder']/'predictions.npz') as p:one=p['test_pred']
            with np.load(DATA/valid[1]['candidate_folder']/'predictions.npz') as p:two=p['test_pred']
            indices=np.flatnonzero(one!=two)
            if len(indices):add(indices[0],'First disagreement between first two readable model submissions')
    for i in range(len(dataset['test'])):
        if len(example_ids)>=5:break
        add(i,'First unused official test item to complete five distinct examples')
    examples=[]
    for i,why in zip(example_ids,selection):
        row={'id':f'test-{i:04d}','index':i,'gold':int(dataset['test'][i,-1]),'selection':why,'predictions':{}}
        for r in valid:
            with np.load(DATA/r['candidate_folder']/'predictions.npz') as p:row['predictions'][r['label']]=int(p['test_pred'][i])
        examples.append(row)
    save_json(DATA/'worked_examples.json',examples)
    fig,axes=plt.subplots(1,len(examples),figsize=(12,4))
    for ax,case in zip(np.atleast_1d(axes),examples):
        ax.imshow(dataset['test'][case['index'],:64].reshape(8,8),cmap='Greys',vmin=0,vmax=16);ax.axis('off')
        ax.set_title(f'{case["id"]}\nGold {case["gold"]}',fontsize=10)
        short=lambda n:n.replace(' numbered entries','').replace(' high','').replace(' batch','')
        text='\n'.join(f"{short(name)}: {pred}"+('' if pred==case['gold'] else ' (wrong)') for name,pred in case['predictions'].items()) or 'No scored submission yet'
        ax.text(.5,-.08,text,ha='center',va='top',transform=ax.transAxes,fontsize=9)
    fig.suptitle('Traceable test images expose successes, errors and disagreements');fig.tight_layout()
    fig.savefig(FIGURES/'worked_examples.png',bbox_inches='tight');plt.close(fig)
    if valid:
        lead=sorted(valid,key=lambda r:(-r['test']['accuracy'],r['model_key']))[0];cm=np.asarray(lead['test']['confusion'])
        fig,ax=plt.subplots(figsize=(7,6));im=ax.imshow(cm,cmap='Blues')
        for i in range(10):
            for j in range(10):
                if cm[i,j]:ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8,color='white' if cm[i,j]>cm.max()/2 else '#222')
        ax.set(xticks=range(10),yticks=range(10),xlabel='Predicted digit',ylabel='Gold digit',title=f'{lead["label"]}: which digit classes are confused')
        fig.colorbar(im,ax=ax,label='Number of official test images');fig.tight_layout();fig.savefig(FIGURES/'confusion.png',bbox_inches='tight');plt.close(fig)
    summary={'results':results,'nearest_neighbour':baseline,'legacy_replay':legacy,'readable_submissions':len(valid),
             'unreadable_submissions':sum(not r.get('pending') and r.get('format_failure') is not None for r in results),
             'pending_submissions':sum(bool(r.get('pending')) for r in results),'n_test':1797}
    save_json(DATA/'study_summary.json',summary)
    return summary



import math,re
COUNTS={"W1":4096,"b1":64,"W2":2048,"b2":32,"W3":320,"b3":10}

def strip_fence(text):
    """Remove one Markdown code fence around a reply (```json ... ```). A transport wrapper, never a value change."""
    t=(text or '').strip()
    if t.startswith('```'):
        t=t.split('\n',1)[1] if '\n' in t else ''
        if t.rstrip().endswith('```'):t=t.rstrip()[:-3]
    return t.strip()

def indexed_entries(text):
    """Parse only unambiguous complete numeric members, even with an unfinished outer JSON.

    A numeric token at end of text is not accepted without its following comma/brace,
    because an output limit could have cut the number itself. No values are inferred.
    """
    text=text.strip();decoder=json.JSONDecoder();pos=0;found={k:{} for k in COUNTS};issue=None
    def whitespace(i):
        while i<len(text) and text[i].isspace():i+=1
        return i
    def take(i):return decoder.raw_decode(text,whitespace(i))
    try:
        pos=whitespace(pos)
        if pos>=len(text) or text[pos]!='{':return found,{'kind':'not_json_object','partial':False}
        pos+=1;seen_tensors=set()
        while True:
            pos=whitespace(pos)
            if pos>=len(text):return found,{'kind':'unfinished_outer_object','partial':True}
            if text[pos]=='}':
                pos=whitespace(pos+1)
                assert pos==len(text),'Text after the JSON object'
                return found,None
            name,pos=take(pos);assert isinstance(name,str) and name in COUNTS,'Unknown tensor name'
            assert name not in seen_tensors,'Duplicate tensor object';seen_tensors.add(name)
            pos=whitespace(pos);assert pos<len(text) and text[pos]==':','Expected tensor colon';pos=whitespace(pos+1)
            assert pos<len(text) and text[pos]=='{','Expected numbered tensor object';pos+=1
            while True:
                pos=whitespace(pos)
                if pos>=len(text):return found,{'kind':'unfinished_tensor','tensor':name,'partial':True}
                if text[pos]=='}':pos+=1;break
                index,pos=take(pos)
                assert isinstance(index,str) and re.fullmatch(r'[1-9][0-9]*',index),'Index must be a one-based integer string'
                number=int(index);assert 1<=number<=COUNTS[name],'Out-of-range index'
                assert number not in found[name],'Duplicate indexed entry within reply'
                pos=whitespace(pos);assert pos<len(text) and text[pos]==':','Expected value colon';value_start=whitespace(pos+1)
                value,end=take(value_start);after=whitespace(end)
                if after>=len(text):return found,{'kind':'numeric_token_without_delimiter','tensor':name,'index':number,'partial':True}
                if text[after] not in ',}':
                    tail=text[value_start:].strip()
                    if re.fullmatch(r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]*)?(?:[eE][+-]?[0-9]*)?',tail):
                        return found,{'kind':'unfinished_numeric_literal','tensor':name,'index':number,'partial':True}
                    raise AssertionError('Unexpected character after numerical value')
                assert type(value) in [int,float] and math.isfinite(value),'Non-finite or non-numeric value'
                found[name][number]=value
                if text[after]==',':pos=after+1
                else:pos=after+1;break
            pos=whitespace(pos)
            if pos>=len(text):return found,{'kind':'unfinished_outer_object','partial':True}
            if text[pos]==',':pos+=1
            elif text[pos]=='}':
                pos=whitespace(pos+1);assert pos==len(text),'Text after JSON';return found,None
            else:raise AssertionError('Expected tensor separator')
    except json.JSONDecodeError as exc:
        return found,{'kind':'unfinished_or_malformed_token','partial':True,'position':exc.pos}
    except (AssertionError,ValueError,TypeError,OverflowError) as exc:
        return found,{'kind':'invalid_structure','partial':False,'message':str(exc)}

def missing_ranges(merged):
    result={}
    for name,count in COUNTS.items():
        missing=[i for i in range(1,count+1) if i not in merged[name]];ranges=[]
        for i in missing:
            if ranges and i==ranges[-1][1]+1:ranges[-1][1]=i
            else:ranges.append([i,i])
        if ranges:result[name]=ranges
    return result

def call_records(folder):
    return sorted(folder.glob('turn-*/response.json'))+sorted(folder.glob('call-*/response.json'))

def verify_indexed_artifacts():
    checks=[]
    folders=[f for base in [DATA/'indexed',DATA/'open_models'] if base.exists() for f in sorted(base.iterdir()) if f.is_dir()]
    for folder in folders:
        assembled=folder/'assembled_response.json'
        if not assembled.exists():continue
        merged={k:{} for k in COUNTS};turns=[]
        for path in call_records(folder):
            record=json.loads(path.read_text(encoding='utf-8'));entries,issue=indexed_entries(strip_fence(record.get('answer','')))
            for name,values in entries.items():
                for index,value in values.items():
                    assert index not in merged[name] or merged[name][index]==value,'Conflicting teacher entries'
                    merged[name][index]=value
            turns.append({'path':str(path.relative_to(DATA)),'cache_key':record['cache_key'],'parse_issue':issue})
        assert not missing_ranges(merged),'Assembled submission has missing source indices'
        flat=np.array([merged[name][i] for name,count in COUNTS.items() for i in range(1,count+1)],dtype=float)
        saved=json.loads(assembled.read_text(encoding='utf-8'));payload=json.loads(saved['answer'])
        assert np.array_equal(flat,np.asarray(payload['weights'],dtype=float)),'Assembly changed a source value'
        checks.append({'model_key':folder.name,'values_verified':len(flat),'teacher_values_modified':False,'source_turns':turns})
    save_json(DATA/'indexed_provenance_checks.json',checks)
    return checks

def indexed_workflow_metadata():
    folders=[f for base in [DATA/'indexed',DATA/'open_models'] if base.exists() for f in base.iterdir() if f.is_dir()]
    for folder in folders:
        paths=call_records(folder)
        if not paths:continue
        records=[json.loads(p.read_text(encoding='utf-8')) for p in paths]
        metadata={'started_utc':records[0]['started_utc'],'finished_utc':records[-1]['finished_utc'],
                  'wall_seconds':sum(r['wall_seconds'] for r in records),'usage':{'per_turn':[r.get('usage') for r in records]},
                  'cost_usd':sum(r.get('cost_usd') or 0 for r in records) if all(r.get('cost_usd') is not None for r in records) else None,
                  'turn_cache_keys':[r['cache_key'] for r in records],'outer_requests':len(records),
                  'provider_errors':[r['error'] for r in records if r.get('error') and not (r.get('answer') or r.get('reasoning'))]}
        save_json(folder/'workflow_summary.json',metadata)
        p=folder/'results.json'
        if p.exists() and not (folder/'assembled_response.json').exists():
            result=json.loads(p.read_text(encoding='utf-8'))
            if len(metadata['provider_errors'])==len(records):
                result.update(format_failure=f'{len(records)} provider errors before numerical output',failure_category='provider_unavailable')
            result.update(wall_seconds=metadata['wall_seconds'],usage=metadata['usage'],cost_usd=metadata['cost_usd'])
            save_json(p,result)

def reported_billing():
    batches=[]
    for path in sorted(DATA.glob('**/batch_result.json')):
        if 'legacy' in path.parts:continue
        usage=json.loads(path.read_text(encoding='utf-8')).get('usage') or {}
        if usage.get('cost') is None:continue
        cost=float(usage['cost']);batches.append({'source':str(path.relative_to(DATA)),'cost_usd':cost})
        result_path=path.parent/'results.json'
        if result_path.exists():
            result=json.loads(result_path.read_text(encoding='utf-8'));result.update(cost_usd=cost,cost_source='Cached batch_result.json/usage/cost');save_json(result_path,result)
    indexed=DATA/'indexed/gemini'
    costs=[b['cost_usd'] for b in batches if b['source'].startswith(str(Path('indexed/gemini')))]
    if costs and (indexed/'results.json').exists():
        result=json.loads((indexed/'results.json').read_text(encoding='utf-8'));result.update(cost_usd=sum(costs),cost_source='Sum of indexed per-turn batch billing');save_json(indexed/'results.json',result)
    claude=[]
    for area in ['responses','recovery']:
        path=DATA/area/'opus/response.json'
        if path.exists():claude.append(json.loads(path.read_text(encoding='utf-8')).get('cost_usd') or 0)
    family={}
    for key in FAMILY_LABELS:
        path=DATA/'claude_family'/key/'response.json'
        if path.exists():family[key]=json.loads(path.read_text(encoding='utf-8')).get('cost_usd')
    save_json(DATA/'billing_summary.json',{'google_api_cost_usd':sum(b['cost_usd'] for b in batches),'google_batches':batches,
        'claude_session_list_price_equivalent_usd':max(claude,default=0),
        'claude_family_list_price_equivalent_usd':family,'other_cli_charges_usd':None,
        'nemotron_route':'Free-only with zero-price cap and no paid fallback','note':'API charges and CLI list-price equivalents are different measures.'})

def _offline_network_guard(*args, **kwargs):
    raise RuntimeError("Network requests are disabled in notebook replay; use the explicit collection script for new experiments.")
requests.sessions.Session.request = _offline_network_guard


In [2]:
# Revalidate only cached answers, never collect new ones.
for area in ['responses', 'recovery', 'indexed', 'claude_family', 'open_models']:
    if not (DATA / area).exists():
        continue
    for folder in (DATA / area).iterdir():
        model_key = folder.name.split('-')[0]
        if model_key not in MODELS:
            continue
        cached = folder / ('assembled_response.json' if area in ('indexed', 'open_models') else 'response.json')
        if cached.exists():
            record = json.loads(cached.read_text(encoding='utf-8'))
            validate_and_score(model_key, record, out_override=folder)
summary = make_figures()
print(json.dumps({k:v for k,v in summary.items() if k != 'results'}, indent=2))
print('Fresh teacher calls: 0')
reported_costs = []
claude_cumulative = []
for row in summary['results']:
    value = usage_row(row)['cost_usd']
    if value is not None:
        # The first Opus request and its continuation share one session, whose counter is cumulative.
        first_session = row['model_key'] == 'opus' and not row.get('family') and not row.get('indexed')
        (claude_cumulative if first_session else reported_costs).append(value)
print('Reported USD/list-price equivalents, avoiding duplicate Claude cumulative counters:', round(sum(reported_costs) + max(claude_cumulative, default=0.0), 2))
print('Other CLI dollar charges: not reported, not assumed zero.')


{
  "nearest_neighbour": {
    "hidden_train": {
      "n": 1775,
      "correct": 1749,
      "accuracy": 0.9853521126760564
    },
    "test": {
      "n": 1797,
      "correct": 1750,
      "accuracy": 0.9738452977184195
    }
  },
  "legacy_replay": [
    {
      "model": "astra",
      "parameters": 17,
      "classification_n": 50000,
      "correct": 50000,
      "accuracy": 1.0,
      "regression_n": 50401,
      "regression_mae": 0.0
    },
    {
      "model": "nemotron-no-reasoning",
      "parameters": 17,
      "classification_n": 50000,
      "correct": 24952,
      "accuracy": 0.49904,
      "regression_n": 50401,
      "regression_mae": 0.3347491920881353
    },
    {
      "model": "astra",
      "parameters": 388,
      "classification_n": 20000,
      "correct": 20000,
      "accuracy": 1.0
    },
    {
      "model": "nemotron-free",
      "parameters": 388,
      "classification_n": 20000,
      "correct": 19286,
      "accuracy": 0.9643
    }
  ],
  "readable_subm

Reported USD/list-price equivalents, avoiding duplicate Claude cumulative counters: 41.63
Other CLI dollar charges: not reported, not assumed zero.


## 1. Replication scope and data

The original blog distinguishes reading behaviour from weights, repairing a trained network, and writing a network from scratch. Our direct-writing tests target the last of these. Its tool-free game-writing experiment uses a 600→32→32→6 tanh network, which contains 20,486 weights and biases by calculation from the published shape. Our seventeen-, 388- and 6,570-value MLPs are about 1,205, 53 and 3 times smaller, with different tasks and activations, so they test the mechanism at smaller sizes rather than reproduce the published scores.

| Blog capability | What our programme tests | Replication status |
|---|---|---|
| Write weights from a task specification | Known-rule MLPs and sample-based digit MLPs, with complete numeric output and no teacher tools | Functional small-network examples reproduced on our tasks |
| Repair useful pretrained behaviour | SQL revisions start from an untrained tiny sequence scaffold; frozen word vectors supply representations | Pretrained-network repair claim not tested equivalently |
| Identify a hidden behaviour from weights | The tester inspects submitted matrices after observing outputs | No blind weight-reading experiment conducted |

The positive claim supported here is that language-model-authored numbers can implement useful functions without a tester-run optimizer; there is no matched trained-MLP baseline. The programme so far:

| Experiment | Network size | Observed outcome |
|---|---|---|
| Pretrained reference | 19,969 stored | MacroStories inspected; no preserved scored generation |
| Random language control | 10,000 editable | Ten untrained tokens; near-uniform scores |
| Sol SQL writing and revision | 2,208 editable + 55,500 frozen | Constant query: 1/8 exposed match; revision 0/12 |
| Astra SQL writing | 2,208 editable + 55,500 frozen | Completed run 0/16; earlier cancellation unscored |
| Free-model SQL writing | 2,208 editable + 55,500 frozen | Ultra malformed; Super 6 nonzeros, 0/16 |
| SQL decoding controls | 2,208 editable, unchanged | Zero schema matches in 1,845 outputs, 11 conditions |
| Bounded SQL weights | 2,208 editable + 55,500 frozen | All numerical checks pass; 0/16 SQL matches |
| Weight inspection | 2,208 SQL; 388 MLP; Pythia-14m reference | SQL score collapse; interpretable ring-network structure |
| Known-rule tiny MLPs | 17 per task | Astra 50,000/50,000 classification; regression error zero |
| Four-ring MLP | 388 | Astra 20,000/20,000; Nemotron 96.43%; boundary errors remain |
| Sample-based digit MLP, twelve models | 6,570 | First requests: Astra 91.54%, Fable 90.60%, Sol 79.13%, Muse 33.72%; open-weight 10.8% to 21.1%; six unusable |
| Numbered entries and continuation | 6,570 | Opus 87.92%, Sonnet 67.56%, Gemini 60.71%; Nemotron timeouts; Haiku declined |

The [full ledger](data/replication_ledger.json) adds scope, interruptions and provider failures.

The source is [Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), by E. Alpaydin and C. Kaynak, donated to UCI on June 30, 1998. UCI publishes 5,620 cases with sixty-four integer features. The data are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The archive, its files and their SHA-256 hashes are saved in `data/source/` and the manifest.

Each image is an eight-by-eight grid read row by row, with ink counts from 0 (blank) to 16. The final CSV entry is the class label, which is removed before inference.

The official files provide 3,823 training cases and 1,797 test cases; the test file stays intact. With the fixed seed 20261103, the visible subset has 205 training images for each of digits zero to seven and 204 each for eight and nine, 2,048 in all. The other 1,775 training cases form a secondary hidden split.

Keeping the official test split avoids choosing favourable cases after observing weights. The source notes say thirty writers supplied training images and thirteen different writers supplied test images, so the unused training images are reported as a separate check.

![One actual visible image per digit, with stable official training-row IDs](figures/visible_digits.png)

The ten panels show actual visible cases, one per label, with greyscale for ink counts and the original zero-based training row in each title. A broad loop, a thin vertical stroke and a diagonal stroke leave different patterns, but at this size some strokes merge. Every teacher receives these pixels as text, not as an image.

Every first request uses the same approximately 305 KB packet and prompt hash. No teacher receives a trained network, hidden labels, nearest-neighbour predictions or a test score. No official test pixel row exactly duplicates a visible image, which excludes literal copying but not similar handwriting or pretraining exposure.

## 2. The fixed network and numerical contract

The network has sixty-four inputs, hidden layers of sixty-four and thirty-two neurons, and ten output scores. A fully connected layer multiplies its inputs by a matrix and adds one bias per output neuron. The hidden activation is ReLU, which replaces negative intermediate values by zero. The final layer is linear, so its ten outputs are scores rather than calibrated probabilities.

| Tensor | Shape | Values | Zero-based offsets in the response |
|---|---|---:|---|
| W1 | 64 × 64 | 4,096 | 0–4,095 |
| b1 | 64 | 64 | 4,096–4,159 |
| W2 | 32 × 64 | 2,048 | 4,160–6,207 |
| b2 | 32 | 32 | 6,208–6,239 |
| W3 | 10 × 32 | 320 | 6,240–6,559 |
| b3 | 10 | 10 | 6,560–6,569 |

Matrix rows are output neurons and columns are input neurons. Each tensor is flattened row by row and the tensors are joined in the table's order, so counting and indexing become part of the output task: a correct idea with a missing entry is not an executable submission.

The owner later asked that replies with parsing errors be requested again as one-based numbered objects per tensor, so a model can see where each tensor ends, with a continuation prompt when output size cuts a reply short. The workflow keeps only complete entries and asks only for missing ranges, at most three calls per model. Gemini covers all 6,570 entries in two replies, and Opus 5.5 and Sonnet 5.5, resuming their own CLI sessions, in one each. Nemotron's requests time out upstream and Haiku 5.5 declines. The three open-weight models get this format from their first call and each completes it within two calls.

```python
import numpy as np
def preprocess(raw_pixels):
    # Exactly 64 integers, row-major flattening of the 8 by 8 image.
    # 0 is blank, 16 is the largest ink count. No fitted transform.
    return np.asarray(raw_pixels, dtype=np.float64) / 16.0
def network(raw_pixels, W1, b1, W2, b2, W3, b3):
    x = preprocess(raw_pixels)
    h1 = np.maximum(x @ W1.T + b1, 0.0)
    h2 = np.maximum(h1 @ W2.T + b2, 0.0)
    return h2 @ W3.T + b3
def postprocess(scores):
    # Digit labels are 0 through 9. Ties choose the lowest index.
    return np.argmax(scores, axis=-1)
```

This is the exact code in the teacher packet. Dividing by sixteen is a fixed scale conversion that uses no estimated statistics or labels, and the postprocessor only picks the largest of the ten scores. In an illustrative case where digit five scores highest and eight second, the prediction is five; the gap between the two is a margin, which this notebook uses to pick a difficult example but never reads as a probability.

The flat contract asks for only `{"weights":[...]}` with exactly 6,570 finite JSON numbers. The program never pads a short vector, cuts a long one or replaces values, because such repairs would make the tester a second parameter author.

Google first rejected the exact-length JSON grammar before generating a token; a simpler schema then exhausted a 32,768-token response, and a low-effort variant returned 6,013 numbers. The first Opus continuation returned a refusal citing a provider safety classifier. These are completion and counting outcomes, not digit-classification errors.

A parsed vector is reshaped in float64, saved in `candidate.npz`, checked against the reply and run through the frozen forward code. Sparse networks and zero biases are allowed. A network can pass the contract and still be a poor classifier, which is reported separately.

## 3. Model access and isolation

| Teacher | Pinned version or runtime ID | Provider and call | Published size/precision | Output settings | Why included |
|---|---|---|---|---|---|
| GPT-6 Astra | gpt-6-astra; revision not exposed | OpenAI, codex exec | Not disclosed | xhigh; legacy 17-value medium; no tools | Requested author; known-rule successes |
| GPT-6.1 Sol | gpt-6.1-sol; revision not exposed | OpenAI, codex exec | Not disclosed | xhigh; tools disabled | Requested second OpenAI numerical author |
| Claude Opus 5.5 | claude-opus-5-5; revision not exposed | Anthropic, claude -p | Not disclosed | xhigh with 64,000-token allowance first; high continuation; separate high request with 128,000 tokens; tools disabled | Requested independent vendor; the high-effort request checks whether the first failure was an output-budget effect |
| Claude Fable 5.1 | claude-fable-5-1; revision not exposed | Anthropic, claude -p | Not disclosed | high; 128,000-token allowance; tools disabled | Requested; completes the current Claude family |
| Claude Sonnet 5.5 | claude-sonnet-5-5; revision not exposed | Anthropic, claude -p | Not disclosed | high; 128,000-token allowance; tools disabled | Requested; mid-size Claude model |
| Claude Haiku 5.5 | claude-haiku-5-5; revision not exposed | Anthropic, claude -p | Not disclosed | high; 128,000-token allowance; tools disabled | Requested; smallest and fastest Claude model |
| DeepSeek V4.1 Flash 1017 | deepseek/deepseek-v4.1-flash-20261017; weights deepseek-ai/DeepSeek-V4.1-Flash-1017 | DeepSeek, OpenRouter, no fallback | 763B stored values in the published safetensors (MIT); serving precision not stated | medium reasoning; 131,072-token allowance; numbered entries from the start | Highest open-weight model on the Artificial Analysis Progress Index (324) that OpenRouter serves |
| GLM-5.3 | z-ai/glm-5.3-20260816; weights zai-org/GLM-5.3 | Z.AI, OpenRouter, no fallback | 753B stored values, mostly fp8 (GLM-5.3 licence); served in fp8 | medium reasoning; 131,072-token allowance; numbered entries from the start | Second-highest open-weight model on the same index (323) |
| MiMo-V2.6-Pro | xiaomi/mimo-v2.6-pro-20260921; weights XiaomiMiMo/MiMo-V2.6-Pro-RL | Xiaomi, OpenRouter, no fallback | 1.02T stored values, mostly packed 8-bit (MIT); served in fp8 | reasoning on (no effort setting); 131,072-token allowance; numbered entries from the start | Third open-weight model on the same index (321) that OpenRouter serves with weights |
| Muse Wonder Full | muse-wonder-full-1.0; revision not exposed | Meta, muse exec | Not disclosed | xhigh; empty preset; tools disabled | Requested independent vendor |
| Nemotron 3.5 Super VL | nvidia/nemotron-3.5-super-vl-20261015 | NVIDIA, OpenRouter free | 120B total, 12B active; precision not attested | Reasoning first; off indexed; 131,072 cap; free only | Requested open-weight author; 388-value comparison |
| Gemini 4 Argon batch | google/gemini-4-argon-20261013 | Google Vertex, OpenRouter Batch API | Not disclosed | high original and schema retry; low final variant; 131,072 requested output cap | Requested Google comparison; asynchronous batch route |

The CLIs do not expose dated weight revisions, so the table says so instead of presenting an alias as a pinned checkpoint. OpenRouter metadata pins canonical dated slugs before requests are sent.

```mermaid
flowchart TD
    P["One frozen packet<br/>2,048 labelled rows + code"] --> A["Astra: codex exec, xhigh"]
    P --> S["Sol: codex exec, xhigh"]
    P --> C["Opus 5.5: claude -p, xhigh, 64k tokens"]
    P --> W["Muse Wonder Full: muse exec, xhigh"]
    P --> N["Nemotron: OpenRouter free"]
    P --> G["Gemini Argon: OpenRouter Batch API, high"]
    P --> FB["Fable 5.1: claude -p, high, 128k tokens"]
    P --> CH["Opus 5.5: claude -p, high, 128k tokens"]
    P --> SN["Sonnet 5.5: claude -p, high, 128k tokens"]
    P --> HK["Haiku 5.5: claude -p, high, 128k tokens"]
    C -->|"Output/budget stop"| CC["Opus: claude -p, high continuation<br/>saved context; refusal"]
    G -->|"Schema rejected before decoding"| GR["Gemini: Batch API, high<br/>simpler transport schema"]
    GR -->|"Thinking/output limit"| GL["Gemini: Batch API, low<br/>same user packet"]
    N -->|"Output limit"| NI["Nemotron: OpenRouter, reasoning off<br/>numbered tensor entries"]
    GL -->|"Wrong count"| GI["Gemini: Batch API, low<br/>numbered tensor entries"]
    CH -->|"Wrong count"| CI["Opus: claude -p resumes its session<br/>numbered tensor entries"]
    SN -->|"Wrong count"| SI["Sonnet: claude -p resumes its session<br/>numbered tensor entries"]
    HK -->|"Zeros to the output cap, then a note"| HI["Haiku: claude -p resumes its session<br/>numbered entries requested; declined"]
    NI -->|"Missing indices; CONTINUE, at most 3 calls"| NI
    GI -->|"Missing indices; CONTINUE, at most 3 calls"| GI
    CI -->|"Missing indices; CONTINUE, at most 3 calls"| CI
    SI -->|"Missing indices; CONTINUE, at most 3 calls"| SI
    P -->|"Return section asks for numbered entries"| P2["Same packet with numbered return section<br/>+ output-budget note"]
    P2 --> DS["DeepSeek V4.1 Flash: OpenRouter, DeepSeek<br/>medium reasoning, 131k tokens"]
    P2 --> GZ["GLM-5.3: OpenRouter, Z.AI<br/>medium reasoning, 131k tokens"]
    P2 --> MM["MiMo-V2.6-Pro: OpenRouter, Xiaomi<br/>reasoning on, 131k tokens"]
    DS -->|"Reasoning only, cut off or not JSON: follow-up, at most 3 calls"| DS
    GZ -->|"Same follow-up rule"| GZ
    MM -->|"Same follow-up rule"| MM
```

Requests are independent and pass nothing between models: no scores, labels or other models' weights. Teachers run in empty working directories with no shell, code execution, file tools, web search, skills or agents; startup records confirm empty tool catalogues, and no tool-use event occurred. Provider scaffolding, output enforcement and reasoning-effort labels differ, which is part of this practical comparison.

## 4. How the study got here

The earlier tiny MLP had two inputs, four ReLU hidden neurons and one output, seventeen parameters, and a complete rule: class one when two numbers' signs differ, or compute `abs(x1) + 0.5*abs(x2)`. Astra's numbers achieved 50,000/50,000 correct classifications and zero error on 50,401 regression points.

One saved measured example is `(-0.8, 0.3)`, the first example in the archived seventeen-parameter results. Opposite signs require class one. Astra's score is 0.6000000000000001, above the fixed zero threshold, so its class is one. The regression target is `0.8 + 0.5*0.3`; both the cached target and its output are 0.9500000000000001 in float64.

The next network had two hidden layers of sixteen neurons and four outputs, for 388 parameters. Its complete rule assigned points in the unit disk to four rings using radius boundaries 0.25, 0.50 and 0.75. Astra classified all 20,000 balanced random points correctly; free Nemotron classified 19,286/20,000 correctly, or 96.43%. On a dense grid Astra scored 125,601/125,625, with the errors at exact boundaries.

The Astra ring network reads as an approximation to radius followed by thresholds, so its many zeros have a functional explanation; this is why the digits experiment allows sparse constructions.

![Cache-only reproduction of the earlier known-rule classification trials](figures/legacy_accuracy.png)

The two panels show different tasks, not a size-controlled comparison. The seventeen-value task has two labels and a 50% chance line; the ring task has four labels and a 25% chance line. Astra reaches the measured ceiling in both. The notebook recomputes these counts from the arrays archived under `data/legacy/`. These trials are not rows of the digit comparison, which adds sixty-four visual features, ten labels and a sample in place of a complete rule.

The digit comparison began with six models. The first Opus 5.5 request spent its whole output allowance on thinking, so the owner asked for weights from Opus, Sonnet 5.5 and Haiku 5.5, and later Fable 5.1, with more room to write. Claude Code subagents could not hold the 268,000-token packet, so the requests went through `claude -p` at high effort with a 128,000-token allowance. The owner then asked for the three highest-ranked open-weight models that OpenRouter serves, chosen by the Artificial Analysis Progress Index: DeepSeek V4.1 Flash 1017, GLM-5.3 and MiMo-V2.6-Pro. Because most earlier failures came from running out of output or miscounting, their packet asks for numbered entries from the first call and says that the entries alone take about 45,000 tokens; it differs from the other packets only in that return section. The [protocol](data/PROTOCOL.md) records each step.

## 5. Results and comparisons

A blank reply, unfinished stream or malformed vector is not a set of digit predictions, so those submissions get their own table with the reason instead of an accuracy.

| Submission | Workflow | Official test | Wilson 95% | Visible accuracy | Unused training |
|---|---|---|---|---|---|
| Astra | First request | 1,645/1,797 (91.54%), 152 wrong | 90.16–92.74% | 93.16% | 94.03% |
| Sol | First request | 1,422/1,797 (79.13%), 375 wrong | 77.19–80.95% | 82.91% | 83.61% |
| Muse Wonder | First request | 606/1,797 (33.72%), 1191 wrong | 31.57–35.94% | 37.74% | 37.86% |
| Fable 5.1 | Claude request, high effort | 1,628/1,797 (90.60%), 169 wrong | 89.16–91.86% | 90.82% | 91.83% |
| DeepSeek V4.1 Flash | Numbered from the start, 1 call | 379/1,797 (21.09%), 1418 wrong | 19.27–23.04% | 19.73% | 21.35% |
| GLM-5.3 | Numbered from the start, 2 calls | 194/1,797 (10.80%), 1603 wrong | 9.44–12.32% | 11.18% | 10.54% |
| MiMo-V2.6-Pro | Numbered from the start, 2 calls | 335/1,797 (18.64%), 1462 wrong | 16.91–20.51% | 18.75% | 20.00% |
| Gemini batch numbered entries | Numbered entries, 2 calls | 1,091/1,797 (60.71%), 706 wrong | 58.43–62.95% | 61.82% | 63.83% |
| Opus 5.5 high numbered entries | Numbered entries, 1 call | 1,580/1,797 (87.92%), 217 wrong | 86.34–89.35% | 88.57% | 88.68% |
| Sonnet 5.5 numbered entries | Numbered entries, 1 call | 1,214/1,797 (67.56%), 583 wrong | 65.36–69.68% | 69.48% | 70.54% |

These submissions produced no network to score:

| Submission | Workflow | Why no network | Unusable replies |
|---|---|---|---|
| Opus 5.5 | First request | No weights: reasoning used the whole output allowance | 1 unusable reply |
| Nemotron | First request | No weights: output limit reached during reasoning | 1 unusable reply |
| Gemini batch | First request | No weights: provider rejected the output schema | 1 unusable reply |
| Opus 5.5 high | Claude request, high effort | Wrong count: 6,378 of 6,570 values | 1 unusable reply |
| Sonnet 5.5 | Claude request, high effort | Wrong count: 6,624 of 6,570 values | 1 unusable reply |
| Haiku 5.5 | Claude request, high effort | No weights: two runs of zeros hit the output cap, then a note | 1 unusable reply |
| Opus 5.5 continuation | Recovery | No weights: refusal, citing a provider safety classifier | 1 unusable reply |
| Gemini batch schema retry | Recovery | No weights: output limit reached during reasoning | 1 unusable reply |
| Gemini batch low effort | Recovery | Wrong count: 6,013 of 6,570 values | 1 unusable reply |
| Nemotron numbered entries | Numbered entries, 3 calls | No weights: three provider timeouts | 3 provider errors |
| Haiku 5.5 numbered entries | Numbered entries, 1 call | No weights: declined to write untested weights | 1 unusable assembly |

![Official test accuracy for every submission, grouped by how it was requested, with the reason for each failure](figures/test_accuracy.png)

Rows are grouped by how each submission was requested. Blue bars show accuracy on the 1,797 official test images; a cross marks a submission with no network, which is not an accuracy of zero. The brown bar is the nearest-neighbour reference and the dashed line is random guessing. Error bars are Wilson 95% intervals over test images for each fixed network; they do not cover what another request would produce.

The highest first-request score is Astra's, 1,645/1,797 (91.54%); the nearest-neighbour reference labels 105 more images correctly. Astra and Fable 5.1, both first requests, differ by 17 images. Three networks come from numbered-entry retries and are not first-reply successes: Opus 5.5 high 1,580, Sonnet 5.5 1,214, Gemini batch 1,091. All of these are differences between single submitted networks, not a stable ranking across repeated requests or new datasets.

Fable 5.1's first reply is the second-best network, 17 correct images behind Astra. Opus 5.5's numbered network follows, 48 behind Fable and 158 ahead of Sol. It took two calls, a flat reply with three missing rows and then a numbered reply, so it is not a first-reply success. Sonnet 5.5's network is right on 1,214 images. It recognises zeros and sixes well (94% and 97% of them) but fewer than half of the fours and fives (41% and 45%). Section 7 shows that the Claude networks are template matchers.

The three open-weight networks sit near the random-guessing line: DeepSeek V4.1 Flash 379 correct, MiMo-V2.6-Pro 335 and GLM-5.3 194, against about 180 expected from guessing. Their numbered replies parsed without a single conflict or gap, so these are wrong digits, not format failures; section 6 shows why.

![Output tokens per submission, split into reported reasoning and visible reply](figures/output_tokens.png)

Each bar is one submission's output in tokens: light for reasoning the provider counted but did not show, solid for the visible reply, blue when a network resulted. Submissions that produced a network used 30,414 to 181,399 output tokens over their calls. Fable 5.1 and GLM-5.3 are at the top because their thinking hit the output cap once before they wrote; the first Opus request, Nemotron and the Gemini schema retry hit it and wrote nothing. Opus 5.5 high and Sonnet 5.5 finished well inside 128,000 tokens and still miscounted, so a larger allowance alone did not fix them; numbered entries did. Haiku's flat reply ran to 256,483 tokens, almost all zeros. Tokenizers differ, so lengths compare only approximately.

No scored network does much better on the visible sample than on hidden images: each one's visible, unused-training and test accuracies lie within about five percentage points. The nearest-neighbour reference (97.38% on test, 98.54% on unused training) stores all 2,048 visible images and compares them at prediction time, a lookup the MLP does not have, so it shows useful structure in the sample rather than a target for a tool-free author.

## 6. Worked test cases and error analysis

The cases follow fixed rules, so striking examples cannot be picked by hand: the first test row, the first success and first error of the highest-scoring network, its smallest top-two margin, and the first unused rows to make five. Row IDs connect each case to `data/source/optdigits.tes` and the saved predictions.

![Five actual official test images with gold labels and predictions from available networks](figures/worked_examples.png)

### Case 1: `test-0000`, gold digit 0

Selection: First official test item, fixed ordinary case. 7 of 10 networks correct; wrong: Muse Wonder → 8, DeepSeek V4.1 Flash → 3, MiMo-V2.6-Pro → 1.

### Case 2: `test-0002`, gold digit 2

Selection: First error for highest observed score. 1 of 10 networks correct; wrong: Astra → 8, Sol → 1, Muse Wonder → 1, Fable 5.1 → 1, DeepSeek V4.1 Flash → 1, GLM-5.3 → 0, MiMo-V2.6-Pro → 1, Gemini → 1, Sonnet 5.5 → 1.

### Case 3: `test-0415`, gold digit 9

Selection: Smallest uncalibrated top-two score margin. 4 of 10 networks correct; wrong: Astra → 3, Muse Wonder → 3, DeepSeek V4.1 Flash → 1, GLM-5.3 → 0, MiMo-V2.6-Pro → 1, Sonnet 5.5 → 8.

### Case 4: `test-0001`, gold digit 1

Selection: First unused official test item to complete five distinct examples. 9 of 10 networks correct; wrong: GLM-5.3 → 0.

### Case 5: `test-0003`, gold digit 3

Selection: First unused official test item to complete five distinct examples. 4 of 10 networks correct; wrong: Sol → 8, Muse Wonder → 1, DeepSeek V4.1 Flash → 1, GLM-5.3 → 0, MiMo-V2.6-Pro → 1, Sonnet 5.5 → 5.

### End-to-end calculation for `test-0000`

The first eight raw ink counts are `[0, 0, 5, 13, 9, 1, 0, 0]`; dividing by sixteen gives `[0.0, 0.0, 0.3125, 0.8125, 0.5625, 0.0625, 0.0, 0.0]`, and the other 56 inputs are scaled the same way. Astra's unchanged matrices yield these ten scores:

0: 86.5 · 1: 67.0625 · 2: 72.875 · 3: 73.5625 · 4: 73.5 · 5: 76.0625 · 6: 74.25 · 7: 67.6875 · 8: 75.0625 · 9: 78.6875

The largest score has digit 0; the hidden label is 0, so this image adds 1 to the network's correct count. Nothing is fed back into its parameters.

The panels show the actual sixty-four-value inputs before division by sixteen; the gold labels were absent from the packet. In case 2, Opus's template network labels a two correctly while the other eight networks call it an eight, a one or a zero. GLM-5.3 answers 0 in all five cases, the collapse described below.

![Confusion matrix for highest observed score](figures/confusion.png)

Rows are gold digits and columns are predictions. Each cell counts official test images, and the diagonal counts correct answers. For Astra, 13 of 180 true digit-9 images were labelled 5; 12 of 180 true digit-9 images were labelled 3. These are the two largest off-diagonal counts; they show which confusions recur, not why.

### Replies that failed the count

Two unusable Claude replies show that a format failure can hide a working design. Sonnet 5.5's flat reply (`data/claude_family/sonnet/weights_output.json`) puts one matrix row per line, so its layout can be read without changing a value:

| Lines of the reply | Values per line | What the contract needs there |
|---|---|---|
| 1 to 64 | 64 | W1, 64 rows of 64 |
| 65 | 64 | b1, 64 values |
| 66 | 56, all zeros | nothing |
| 67 to 98 | 64 | W2, 32 rows of 64 |
| 99 | 30 | b2, 32 values |
| 100 to 109 | 32 | W3, 10 rows of 32 |
| 110 | 10 | b3, 10 values |

The extra line adds 56 values and the short bias line removes 2, so the vector has 6,624 entries and every tensor after b1 would be read from the wrong offsets. Picking which line to drop would make the tester a co-author, so the reply stays unusable. Sonnet's numbered reply has the same templates and biases without those two errors and scores 1,214/1,797. Opus 5.5's flat reply lacks two first-layer and one second-layer row, all three zeros; its numbered reply repeats the same nonzero rows and scores 1,580/1,797.

Haiku 5.5 is the opposite case. Every number after the 491st in its flat reply is zero; it hit the output cap, restarted when the CLI asked it to resume, hit it again and ended with a note that its attempts had "degenerated into long runs of zeros". Asked for numbered entries, it declined to write weights it could not test and offered a training script, which the protocol does not allow.

### Networks that collapse onto one or two digits

The three open-weight networks are the opposite of the Claude count failures: the format is perfect and the digits are wrong. Each one sends most images to one or two classes.

| Network | Design | Most-predicted digits (of 1,797) | What tips the scores |
|---|---|---|---|
| DeepSeek V4.1 Flash | 10 hand-drawn templates in layer 1, passed straight through | 1: 1,388; 4: 199 | The "1" template rewards central ink that most digits have (mean activation 0.39, against 0.11 to 0.31) |
| GLM-5.3 | Identity layer 1; 10 digit-average templates in raw ink counts in layer 2 | 0: 1,670; 3: 127 | The "0" template holds 374 units of ink, against 175 to 266, while the per-digit offsets differ by only 4.5; the templates for 4 to 9 also do not resemble the real digit averages |
| MiMo-V2.6-Pro | Identity layer 1; 32 region sums; dense output layer | 1: 1,546; 7: 171 | The output offsets run from −62.2 for digit 0 to −21.5 for digit 1, a 40-point head start for "1" |

GLM's templates for 0 to 3 resemble the real averages of those digits in the sample (correlation 0.57 to 0.89), and those for 1 to 3 match the last versions in its reasoning entry for entry. Its templates for 5 to 9 match its reasoning best when shifted three or four positions, which suggests the values slid against their index numbers as it wrote them; it revised the templates many times, so this is a likely reading rather than a proof. Fable 5.1 built the same kind of design with templates that match the real averages (correlation 0.96 to 0.99) and per-digit biases that balance them, and scored 90.60%. What the three open-weight networks lack is the step that makes ten scores comparable, such as subtracting half of each template's squared length in a nearest-average classifier. A training run would tune that automatically; a model writing numbers by hand has to work it out, and none of these three did.

## 7. Reasoning logs and parameter patterns

Provider-emitted reasoning can show the construction a teacher intended, though it need not expose every step and a plausible plan does not validate the numbers. Nemotron ([160,543 characters](data/responses/nemotron/reasoning.txt)) and the three open-weight models stream readable reasoning; the Claude CLI and Codex report reasoning-token counts with no text. Nothing absent is reconstructed.

Many zeros can mean unused neurons, a deliberately sparse program or a constant predictor, so a weight pattern only counts as explained when it is tied to the forward computation or the predictions.

| Submission | Exact zeros / 6,570 | Nonzero first-layer rows / 64 | Nonzero second-layer rows / 32 | Largest magnitude |
|---|---|---|---|---|
| Astra | 5344 | 40 | 20 | 40.625 |
| Sol | 5546 | 40 | 20 | 19.25 |
| Muse Wonder | 6257 | 10 | 10 | 12.75 |
| Fable 5.1 | 4803 | 64 | 30 | 109.5 |
| DeepSeek V4.1 Flash | 5900 | 10 | 10 | 1 |
| GLM-5.3 | 6174 | 64 | 32 | 28.3867 |
| MiMo-V2.6-Pro | 5984 | 64 | 32 | 62.208 |
| Gemini batch numbered entries | 5438 | 64 | 20 | 2.7 |
| Opus 5.5 high numbered entries | 6005 | 13 | 13 | 18.2617 |
| Sonnet 5.5 numbered entries | 6097 | 64 | 10 | 15.23 |

![Nonzero coefficients by layer for every scored submitted network](figures/parameter_use.png)

The stacked bars count nonzero coefficients in each affine layer, including its bias, against the 6,570 values every network supplies (dashed line). A nonzero row is a structural count, not proof that the neuron activates on an image, and more nonzero entries do not make a better classifier: Opus's network uses 565 nonzero values and scores higher than Gemini's 1,132.

Row counts alone cannot show that fewer neurons caused a lower score: Opus uses thirteen rows in each hidden layer and scores 87.92%, Muse ten and 33.72%.

The Claude networks are template matchers, a design that compares an image with a stored typical image of each digit. Fable's identity first layer feeds ten second-layer templates that correlate 0.96 to 0.99 with the real digit averages, each with its own bias, plus 20 correction units that switch on for 1% to 9% of test images and add to one digit's score. Read from the submitted weights, the templates with their biases alone label 1,614 test images correctly, 194 more than without the biases, and the correction units add 14. Sonnet's identity first layer hands the 64 scaled pixels to the second layer, where each of the first ten rows is a digit template with a bias near 14; the output layer copies template c to digit c. Each digit score is therefore one template's dot product with the image plus a bias. Opus's thirteen first-layer rows are templates with biases between 15 and 18, large enough that no template unit switches off on any test image (smallest pre-activation 26.2). Digits 1, 7 and 9 get two templates each. The second layer computes `ReLU(b - a)` beside `a`, and the output layer adds them, which equals `max(a, b)` for nonnegative `a` and `b`. On all 1,797 test images, scoring each digit by its best template reproduces Opus's full network exactly, and the one-template-per-digit reading of Sonnet's network matches its full scores exactly. This reads the submitted weights, not hidden reasoning.

The open-weight models' reasoning text shows where their designs came from. DeepSeek V4.1 Flash decided to "handcraft digit templates" from common shapes instead of computing them from the 2,048 images, and spent much of its 45,801 characters planning how to write thousands of zero entries without losing count. GLM-5.3 spent 312,221 characters estimating and revising an average image for each digit from the sample until its allowance ran out; its second call wrote those estimates as templates. MiMo-V2.6-Pro planned sums over rows, columns and regions, and its first call had begun listing entries inside its reasoning when the provider stream failed.

Astra and Sol have identical second-layer and output matrices and biases, so their different predictions come from the first layer. For each digit, group four first-layer ReLU outputs as `a,b,c,d`: the second layer computes `ReLU(a+b-c-d)` and `c+d`, and the output layer adds them, which equals `max(a+b, c+d)` because all four are nonnegative.

The notebook checks each of these readings against the full network on all 1,797 test images (`data/program_structure.json`). They explain why dozens of hidden slots can stay unused in a working classifier, and they never change a coefficient.

## 8. Limits and practical meaning

The strongest data limitation is the benchmark's age: these files were donated in 1998, so a model may have seen the data, class prototypes or code for it. A fresh digit collection is needed before claiming learning from an unseen dataset.

The strongest design limitation is one generated network per model and setting. Image-level intervals hold that network fixed and say nothing about how much a second request would differ; a capacity comparison would repeat requests and visible samples and summarize the spread across networks.

Accuracy weights digits by their frequency in the official test split, so per-class results matter: Sonnet's network, for example, gets fewer than half of the fours and fives right. This is a single small-image dataset, and the scores do not transfer to higher-resolution images or document OCR.

No optimizer is run on this architecture, so its best achievable accuracy is unknown; the nearest-neighbour reference shows useful information in the sample, not an MLP ceiling.

Output length and counting are part of the challenge: 6,570 explicit numbers take about 30,000 output tokens before any reasoning. Numbered entries fixed both Claude counts and let all three open-weight models finish, but only the open-weight models had them from the start, so this design cannot say how the others would do with them from the start or with matched settings.

All results concern numerical writing with teacher tools disabled; allowing execution or fitting, as some of the blog's settings do, is a different workflow.

## 9. Reproduction, caches and spend

The executed [notebook](41_direct_weight_mlps.ipynb) defines its helpers internally, blocks network requests, and replays local caches to reproduce predictions, figures and reported spend. Initial requests are in `data/responses/`, earlier recoveries in `data/recovery/`, the high-effort Claude requests in `data/claude_family/`, numbered retries in `data/indexed/` and the open-weight requests in `data/open_models/`. Source images, sample membership and hidden labels are committed for audit.

Install Python with NumPy, SciPy, matplotlib, requests, nbformat and nbconvert. From this folder, run `jupyter nbconvert --to notebook --execute --inplace 41_direct_weight_mlps.ipynb`; replay needs no model credential. `indexed_recovery.py` runs the numbered-entry workflow for OpenRouter models; `claude_family.py` and `claude_indexed.py` collect the Claude requests and their numbered replies, and `open_models.py` the open-weight requests.

| Submission | Input tokens | Output tokens | Of which reasoning | Wall time | Reported USD or list-price equivalent |
|---|---|---|---|---|---|
| Astra | 270,858 | 30,414 | 16,471 | 546 s | Not reported |
| Sol | 270,858 | 35,562 | 21,633 | 530 s | Not reported |
| Opus 5.5 | 268,014 | 64,000 | 64,000 | 511 s | $5.60 |
| Muse Wonder | 282,926 | 30,415 | 16,786 | 424 s | Not reported |
| Nemotron | 302,686 | 131,072 | 131,072 | 2,114 s | $0.00 |
| Gemini batch | Not reported | Not reported | Not reported | 371 s | $0.00 |
| Fable 5.1 | 536,090 | 181,399 | 166,646 | 1,418 s | $19.79 |
| Opus 5.5 high | 268,014 | 42,130 | 28,224 | 330 s | $2.99 |
| Sonnet 5.5 | 268,020 | 28,015 | 13,582 | 160 s | $1.35 |
| Haiku 5.5 | 1,188,321 | 256,483 | 23,003 | 717 s | $1.20 |
| DeepSeek V4.1 Flash | 267,245 | 50,911 | 17,821 | 170 s | $0.07 |
| GLM-5.3 | 665,796 | 176,061 | 142,393 | 1,323 s | $1.40 |
| MiMo-V2.6-Pro | 452,184 | 82,578 | 38,590 | 1,628 s | $0.07 |
| Opus 5.5 continuation | 536,497 | 1,158 | 1,007 | 17 s | $5.74 (session total) |
| Gemini batch schema retry | 302,875 | 32,768 | 32,765 | 710 s | $0.47 |
| Gemini batch low effort | 302,875 | 13,723 | 0 | 610 s | $0.37 |
| Gemini batch numbered entries | 662,718 | 45,237 | 0 | 1,386 s | $0.89 |
| Nemotron numbered entries | Not reported | Not reported | Not reported | 364 s | Not reported |
| Opus 5.5 high numbered entries | 310,840 | 39,095 | 967 | 250 s | $4.16 |
| Sonnet 5.5 numbered entries | 296,731 | 40,573 | 2,366 | 183 s | $1.90 |
| Haiku 5.5 numbered entries | 525,336 | 268 | 0 | 7 s | $1.23 |

Unknown CLI charges are not treated as zero. The first Opus session's counter is cumulative: US$5.74 in all, including the US$5.60 stop. Together these seven Claude calls report US$32.62 in list-price equivalents, US$19.79 of it Fable 5.1. The five open-weight calls cost US$1.54 on OpenRouter, US$1.40 of it GLM-5.3.

The [protocol](data/PROTOCOL.md) records design changes, recovery settings and parser rules, and the [manifest](data/manifest.json) pins inputs and model identities. Replay sends no traces; fresh calls go to a local Langfuse when one is running.

## 10. Future work and further reading

Links below were checked on November 3, 2026.

### Future work

- Reproduce a published game-writing task with matched activations, evaluation and a trained baseline, then reduce network size. Measure which published performance comparisons survive; the current positive results use different tasks.
- Add a blind weight-reading test using matched networks for different known rules. Ask teachers to identify behaviour without outputs, rather than relying on the tester's post-hoc interpretation.
- Start with a competent pretrained small network and a defined defect, then request targeted edits. Measure retained skill and repaired behaviour; our SQL scaffold lacks the trained sequence core used in the blog's repair setting.
- Ask every model for numbered entries from the first request, with matched effort and output allowances, and repeat each request several times. Measure format completion and accuracy; numbered entries fixed both Claude count errors and let all three open-weight models finish, but the open-weight networks still scored 11% to 21%.
- Use newly collected handwriting or a preregistered new labelling and multiple visible samples. Measure variation between submitted networks and reduce the public-data exposure caveat.

### Further reading

- [Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) (Alpaydin and Kaynak, 1998). The source dataset and its official train/test files.
- [Sparks of Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/) (Takehi et al., 2026). The motivating distinction between directly writing numerical networks and editing existing pretrained parameters.
- [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html) (Glorot and Bengio, 2010). Explains scale considerations for network initialization, without making a histogram a test of functional correctness.
- [Git Re-Basin: Merging Models modulo Permutation Symmetries](https://arxiv.org/abs/2209.04836) (Ainsworth et al., 2023). Shows why neuron coordinates and raw parameter patterns need functional interpretation.
- [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html) (Guo et al., 2017). Explains why classifier scores and margins should not be read as reliable confidence probabilities.